# Name That LLM: frozen-LM fingerprints + prompt-group decoding

**Task.** Name which of 18 LLMs wrote each 40-250-word passage. The metric is balanced accuracy. The hidden test set
has about 4000 passages, all answering prompts that do **not** appear in train.

### TL;DR
1. **Exploit the structure.** In train, each of the 150 prompts was answered by every model exactly once. The test set
   is built the same way. We **recover the prompt groups on test by clustering** (topic TF-IDF, average linkage).
   96-98% of passages land in exactly their true group. Then we **decode jointly inside each group**: each model
   appears at most once per group (Sinkhorn), and sibling models write similar answers to the same prompt (a
   pairwise term). Decoding is worth **+0.10 to +0.20** balanced accuracy on top of any classifier.
2. **Prompt-centring.** We subtract the mean of the passage's prompt group from every feature vector. This removes
   the topic and keeps the style. It is the biggest single lever: frozen-LM LDA goes from 0.43 to 0.60.
3. **Unary models.** Each is prompt-centred and has an uncentred twin, which is the fallback for tiny clusters.
   * **A.** TF-IDF char 1-5 + word 1-2 + 442 stylometric features, then a centred-Gram logistic regression, plus a
     multinomial NB. CPU, about 3 minutes.
   * **B.** **Frozen decoder LMs**: Qwen3-4B/1.7B/0.6B-Base. We prepend an `<|endoftext|>` sink token and
     mean-pool 11-14 hidden layers (the sink is excluded), then prompt-centre, then StandardScaler, PCA and
     shrinkage LDA. No fine-tuning. The 4B model uses a hierarchical family/sibling LDA.
   * **C.** **DeBERTa-v3-large**, fine-tuned with whole prompt groups per batch. It has a raw head, a centred head
     and a column (assignment) loss.
4. **Ensemble.** An equal-weight geometric mean of the components, chosen by nested CV. Then Sinkhorn +
   pairwise mean-field decoding inside the predicted clusters.

### Cross-validation (5 GroupKFold folds by `prompt_id`, the same folds everywhere)
All numbers are honest: validation passages are centred by *predicted* clusters, and the pair models are fitted
out-of-fold. "Decoded" means Sinkhorn + pair decoding of that component alone with full groups.

| Component | argmax bacc | decoded |
|---|---|---|
| A. TF-IDF + stylometry LR (centred) + 0.2 MNB | 0.473 | 0.582 |
| B. Qwen3-4B-Base, hierarchical pc-LDA | 0.598 | 0.707 |
| B. Qwen3-1.7B-Base pc-LDA | 0.575 | 0.697 |
| B. Qwen3-0.6B-Base pc-LDA | 0.563 | 0.681 |
| C. DeBERTa-v3-large group fine-tune (centred head) | 0.458 | 0.539 (Hungarian) |
| **Final: equal-weight blend of A + B + C, mix9 fallback, Sinkhorn/pair decoding (x1.5, lam 0.25)** | 0.625 | **0.733** |

Under per-model dropout (unequal per-model counts, as on the test) the final blend scores 0.709 at 20% and 0.687
at 40%. Optional extras measured on top of it: Qwen3-8B-Base +0.002 (needs 2xT4), Qwen2.5-3B-Instruct +0.001,
SmolLM2-1.7B + OLMo-2-1B +0.003 (not hosted on Kaggle). Dropping DeBERTa costs -0.009.
With `RUN_CV=True` the notebook recomputes OOFs for A and B and reports the decoded OOF score of their blend
(section 7).

### Runtime (Kaggle GPU T4, about 4000 test rows)
EDA and clustering take about 1 minute. A takes about 4 minutes, plus 6 minutes for CV. B is GPU feature
extraction (4B about 18, 1.7B about 8, 0.6B about 4 minutes) plus about 10 minutes of CPU LDA including CV.
C takes about 50 minutes. Decoding takes about 2 minutes. **The total is about 1 h 45 min**, well inside the 9 h
limit. A time-budget guard skips any component that would not fit.

### Inputs to attach (Add Input)
| what | where on Kaggle | used by |
|---|---|---|
| competition data | `name-that-llm` (attached automatically) | everything |
| **Qwen3 base** | **Models** -> `qwen-lm/qwen-3`, framework **Transformers**, variations **`4b-base`**, **`1.7b-base`**, **`0.6b-base`** | B |
| **DeBERTa-v3-large** | **Datasets** -> `radek1/deberta-v3-large-hf-weights` (the original HF files; `jonathanchan/deberta-v3-large` also works) | C |

Settings: **Accelerator = GPU T4** (x1 or x2). Do *not* use P100: recent Kaggle PyTorch builds no longer ship
sm_60 kernels, so the GPU components would be skipped. **Internet = Off**. The model scan in section 1 prints
which models it found. A missing model is skipped, and the notebook still submits.

### Robustness
Every component runs inside a guard. A missing model, an OOM or any exception **skips that component**, and the
blend renormalises over what is left. The notebook **always** writes a valid `submission.parquet`. If everything
fails, it falls back to TF-IDF only, and in the worst case to a constant prediction. It runs on the 3-row public
test (tiny or singleton clusters) as well as on the 4000-row hidden test.

## 1. Configuration and environment

Every knob lives in a single `CFG` dict:
* `LOCAL_HOLDOUT` (default `True` when not on Kaggle). We train on folds != `HOLDOUT_FOLD` and use the held-out
  fold's texts as a fake test set, with prompt ids and labels hidden. The full pipeline (clustering, centring,
  decoding) then runs exactly as it will on Kaggle, and the last section reports its balanced accuracy.
* `RUN_CV`: in-notebook grouped OOF for the cheap components (A and the LDA heads of B). It is used to report the
  decoded OOF score and to fit the decoding temperature.
* Component switches, the LM list, ensemble weights, decoding parameters and a **time-budget guard**.

Models are **discovered**, never hard-coded. We walk `/kaggle/input` (depth-limited) for folders that contain a
`config.json` plus weights, then match each wanted model by name patterns. This covers Kaggle Models layouts
(`/kaggle/input/<slug>/transformers/<variation>/<version>/`), dataset or notebook-output layouts
(`/kaggle/input/<ds>/qwen3-4b-base/`) and HF-cache snapshot folders (`models--Qwen--Qwen3-4B-Base/snapshots/<hash>/`).
A model that cannot be found is skipped with a message, and the blend renormalises.

In [ ]:
import os, sys, re, gc, json, glob, math, time, random, warnings, traceback, contextlib
os.environ["HF_HUB_OFFLINE"] = "1"            # no internet calls, ever
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from IPython.display import display

T0 = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/input")

CFG = dict(
    LOCAL_HOLDOUT=not ON_KAGGLE,   # local: train on folds != HOLDOUT_FOLD, "test" = held-out fold texts
    HOLDOUT_FOLD=0,
    HOLDOUT_MAX_DROP=0.0,          # >0: drop held-out passages per model at rates ~U(0, x) (partial groups, like test)
    RUN_CV=True,                   # in-notebook grouped OOF for A and B (CPU, ~15-20 min on Kaggle)
    RUN_EDA=True,
    SEED=42,
    # ---- components -------------------------------------------------------------------------------------
    USE_TF=True,                   # A: TF-IDF + stylometry (CPU)
    USE_LLM=True,                  # B: frozen decoder-LM features (GPU)
    USE_DEBERTA=True,              # C: DeBERTa-v3-large group-aware fine-tune (GPU, ~50 min on T4)
    # Default = only models that can be attached offline from Kaggle Models (qwen-lm/qwen-3). Optional extras
    # (honest CV on top of this set): "q8b" +0.002 (needs 2xT4), "q25i3" (qwen-lm/qwen2.5 3b-instruct) +0.001,
    # "smol17"/"olmo1b" +0.003 together but not hosted on Kaggle (upload them as a dataset yourself).
    LLM_MODELS=["q4b", "q17", "q06"],
    Q4B_HIER=True,                 # hierarchical family -> sibling LDA for Qwen3-4B (+0.5pt argmax, +0.8pt Hungarian)
    ALLOW_CPU_LLM=False,           # run LM feature extraction on CPU when no GPU (very slow; only sensible for q06)
    SKIP_HEAVY_IF_TINY_TEST=False, # True: skip GPU components when the test has < 50 rows (fast commit runs)
    MODEL_PATHS={},                # manual override {key: path}
    # ---- ensemble (work/ens/FINAL_RECIPE.md, honest nested CV): EQUAL-weight mean of the centred log-probs.
    #      Fitted weights did not generalise (nested 0.716 +- 0.014; all candidates within 0.715-0.721).
    #      Missing components drop out of the (normalised) mean.
    ENS_WEIGHTS={"q4b": 1.0, "q8b": 1.0, "q17": 1.0, "q06": 1.0, "smol17": 1.0, "olmo1b": 1.0, "q25i3": 0.5,
                 "tf": 1.0, "deberta": 1.0},
    # ---- small-cluster fallback: rows in a predicted cluster of size m use
    #      alpha * Z_centred + (1 - alpha) * Z_raw, alpha = clip((m - 1) / SMALL_DENOM, 0, 1)   ("mix9")
    #      Z_raw = weighted mean of the uncentred twins below.
    SMALL_DENOM=8,
    RAW_WEIGHTS={"q4b": 1.0, "q8b": 0.5, "q17": 0.5, "smol17": 0.5, "olmo1b": 0.5, "q06": 0.3, "q25i3": 0.3,
                 "tf": 0.3, "deberta": 0.3},
    # ---- decoding (postproc): Sinkhorn per predicted cluster + pairwise sibling term. power 2.05 = 1.5 x the
    #      calibrated 1/T (1.368) of the default 5-component blend (nested CV picked x1.5 / lam 0.25 in 5 of 5
    #      outer folds). power=None -> power_mult * fit_temperature(in-notebook OOF blend).
    DECODE=dict(power=2.05, power_mult=1.5, col="le", pair_lam=0.25, final="argmax", mf_iter=15, mf_damp=0.5),
    CLUSTER=dict(threshold=0.94, w_word=0.5, w_char=0.5),
    # ---- DeBERTa
    DEB_EPOCHS=7, DEB_LR=2e-5, DEB_HEAD_LR=5e-4, DEB_COL_W=0.5, DEB_GROUPS_PER_STEP=2, DEB_KEEP_MIN=0.5,
    DEB_MAX_LEN=512, DEB_EVAL_BS=32, DEB_MAX_STEPS=None,
    # ---- time budget (Kaggle limit 9 h)
    TIME_BUDGET_MIN=500, RESERVE_MIN=12,
)
# optional JSON overrides (used for local smoke tests; unset on Kaggle)
if os.environ.get("NTL_CFG"):
    CFG.update(json.loads(os.environ["NTL_CFG"]))

LABELS = sorted([
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"])
L2I = {l: i for i, l in enumerate(LABELS)}
NC = len(LABELS)
FAMS = [["claude-fable-5", "glm-5.3", "glm-5.3-flash", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5"],
        ["gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash"],
        ["gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra", "qwen3.8-max", "kimi-k3"]]
FAMI = [sorted(L2I[l] for l in f) for f in FAMS]   # sorted == LDA classes_ order


def log(*a):
    print(f"[{(time.time() - T0) / 60:6.1f} min]", *a, flush=True)


SECTION_TIMES = {}


@contextlib.contextmanager
def section(name):
    """Times a block and swallows any exception: one failing component never kills the notebook."""
    t = time.time()
    log(f"=== {name} ===")
    try:
        yield
    except Exception as e:  # noqa
        log(f"!!! {name} FAILED -> skipped: {type(e).__name__}: {e}")
        traceback.print_exc(limit=-8)       # innermost frames: where the error actually happened
    finally:
        SECTION_TIMES[name] = SECTION_TIMES.get(name, 0.0) + time.time() - t
        log(f"=== {name}: {time.time() - t:.1f}s")


def minutes_left():
    return CFG["TIME_BUDGET_MIN"] - (time.time() - T0) / 60


def budget_ok(est_min, what):
    left = minutes_left() - CFG["RESERVE_MIN"]
    if est_min > left:
        log(f"SKIP {what}: needs ~{est_min:.0f} min but only {left:.0f} min of budget left")
        return False
    log(f"{what}: estimated {est_min:.1f} min ({left:.0f} min of budget left)")
    return True


# ---- torch / GPU ----------------------------------------------------------------------------------------
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    N_GPU = torch.cuda.device_count() if HAS_CUDA else 0
    GPU_MEM = [torch.cuda.get_device_properties(i).total_memory / 2**30 for i in range(N_GPU)]
    GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPU)]
except Exception as e:  # noqa
    torch, HAS_CUDA, N_GPU, GPU_MEM, GPU_NAMES = None, False, 0, [], []
    print("torch unavailable:", e)
if HAS_CUDA:
    # is_available() is True even when this torch build has no kernels for the card (e.g. P100 = sm_60 on cu12.8+
    # builds): .to('cuda') works, then the first kernel fails. Probe once and fall back to CPU-only components.
    try:
        _cap = torch.cuda.get_device_capability(0)
        _arch = f"sm_{_cap[0]}{_cap[1]}"
        _archs = torch.cuda.get_arch_list()
        if _archs and _arch not in _archs and f"compute_{_cap[0]}{_cap[1]}" not in _archs:
            raise RuntimeError(f"{_arch} ({GPU_NAMES[0]}) not supported by this torch build {_archs}")
        _a = torch.ones(16, 16, device="cuda", dtype=torch.float16)
        float((_a @ _a).sum().item())
        del _a
    except Exception as e:  # noqa
        print("!!! CUDA is not usable -> GPU components will be skipped. Use the 'GPU T4' accelerator. Reason:", e)
        HAS_CUDA, N_GPU, GPU_MEM, GPU_NAMES = False, 0, [], []
try:
    import transformers
    from packaging import version as _v
    TF_VER = transformers.__version__
    DTYPE_KW = "dtype" if _v.parse(TF_VER) >= _v.parse("4.56.0") else "torch_dtype"
except Exception as e:  # noqa
    transformers, TF_VER, DTYPE_KW = None, None, "torch_dtype"
    print("transformers unavailable:", e)


def seed_everything(s):
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"] = str(s)
    if torch is not None:
        torch.manual_seed(s)
        if HAS_CUDA:
            torch.cuda.manual_seed_all(s)


def free_gpu():
    """Never raises: after a sticky CUDA error, empty_cache() itself would throw outside any guard."""
    gc.collect()
    if HAS_CUDA:
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception as e:  # noqa
            log(f"free_gpu failed: {type(e).__name__}: {e}")


@contextlib.contextmanager
def gpu_section(name):
    """All CUDA work runs in here. Locally (env NTL_GPU_LOCK_DIR) it takes the shared multi-process GPU lock and caps
    memory at 90% (Windows WDDM silently spills to RAM above that); on Kaggle it is a no-op besides freeing memory."""
    cm = contextlib.nullcontext()
    lock_dir = os.environ.get("NTL_GPU_LOCK_DIR")
    if lock_dir and not ON_KAGGLE:
        try:
            sys.path.insert(0, lock_dir)
            from gpu_lock import gpu_lock
            cm = gpu_lock(name)
        except Exception as e:  # noqa
            log("local gpu lock unavailable:", e)
    with cm:
        free_gpu()
        if HAS_CUDA and not ON_KAGGLE:
            try:
                torch.cuda.set_per_process_memory_fraction(float(os.environ.get("NTL_GPU_FRAC", "0.9")))
            except Exception:
                pass
        gpu_mem_report(f"{name} start")
        try:
            yield
        finally:
            gpu_mem_report(f"{name} end (before free)")
            free_gpu()


def gpu_mem_report(tag):
    if not HAS_CUDA:
        return
    try:
        free, tot = torch.cuda.mem_get_info()
        log(f"[gpu] {tag}: device free {free / 2**30:.2f}/{tot / 2**30:.2f} GB | this process: allocated "
            f"{torch.cuda.memory_allocated() / 2**30:.2f} GB, reserved {torch.cuda.memory_reserved() / 2**30:.2f} GB, "
            f"peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
        torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass


seed_everything(CFG["SEED"])
log(f"ON_KAGGLE={ON_KAGGLE} | GPUs={GPU_NAMES} mem={[round(m, 1) for m in GPU_MEM]} GB | torch={getattr(torch, '__version__', None)} "
    f"| transformers={TF_VER} (dtype kw: {DTYPE_KW}) | CPUs={os.cpu_count()}")
print(json.dumps({k: v for k, v in CFG.items()}, indent=None, default=str)[:2000])

### Data
On Kaggle we glob `/kaggle/input/**/test.csv` and keep the folder that also holds `train.csv`. Locally we use the
project `datasets/` folder. Train is sorted by `id`. The 5 folds are a deterministic GroupKFold by `prompt_id`
(shuffled prompts, seed 42). Every OOF number in this project uses these same folds.

In [ ]:
def find_data_dir():
    cands = []
    if os.environ.get("NTL_DATA_DIR"):
        cands.append(os.environ["NTL_DATA_DIR"])
    if ON_KAGGLE:
        cands.append("/kaggle/input/competitions/name-that-llm")
        hits = glob.glob("/kaggle/input/**/test.csv", recursive=True)
        hits = sorted(hits, key=lambda p: ("name-that-llm" not in p.lower(), len(p)))
        cands += [os.path.dirname(h) for h in hits]
    cands += ["datasets", "../datasets", "../../datasets", "data"]
    for c in cands:
        if os.path.isfile(os.path.join(c, "train.csv")) and os.path.isfile(os.path.join(c, "test.csv")):
            return os.path.abspath(c)
    raise FileNotFoundError("train.csv/test.csv not found in " + str(cands))


def harness_folds(df, n_folds=5):
    """Deterministic GroupKFold by prompt_id (identical to the project's common.get_folds)."""
    rng = np.random.RandomState(42)
    prompts = df["prompt_id"].unique()
    perm = rng.permutation(len(prompts))
    pmap = {p: perm[i] % n_folds for i, p in enumerate(prompts)}
    return df["prompt_id"].map(pmap).values.astype(int)


DATA_DIR = find_data_dir()
train_full = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), dtype={"id": str}).sort_values("id").reset_index(drop=True)
train_full["text"] = train_full["text"].fillna("").astype(str)
train_full["y"] = train_full["model"].map(L2I).astype(int)
test_official = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), dtype={"id": str})
test_official["text"] = test_official["text"].fillna("").astype(str)
FOLDS_FULL = harness_folds(train_full)

if CFG["LOCAL_HOLDOUT"]:
    _m = FOLDS_FULL == CFG["HOLDOUT_FOLD"]
    train = train_full[~_m].reset_index(drop=True)
    _ho = train_full[_m].reset_index(drop=True)
    if CFG["HOLDOUT_MAX_DROP"] > 0:
        _rng = np.random.RandomState(CFG["SEED"])
        _rates = _rng.uniform(0, CFG["HOLDOUT_MAX_DROP"], NC)
        _ho = _ho[_rng.uniform(size=len(_ho)) >= _rates[_ho["y"].values]].reset_index(drop=True)
    test = _ho[["id", "text"]].copy()
    Y_HOLDOUT, PID_HOLDOUT = _ho["y"].values, _ho["prompt_id"].values
    folds = FOLDS_FULL[~_m]
    log(f"LOCAL HOLDOUT: train = folds != {CFG['HOLDOUT_FOLD']} ({len(train)} rows), test = fold {CFG['HOLDOUT_FOLD']} "
        f"({len(test)} rows, labels hidden)")
else:
    train = train_full
    test = test_official[["id", "text"]].copy()
    Y_HOLDOUT = PID_HOLDOUT = None
    folds = FOLDS_FULL
y_tr = train["y"].values
pid_tr = train["prompt_id"].values
gtr = pd.factorize(train["prompt_id"])[0]          # integer prompt-group ids of train
tr_texts = train["text"].tolist()
te_texts = test["text"].tolist()
NTR, NTE = len(train), len(test)
FOLD_IDS = sorted(np.unique(folds).tolist())
TINY_TEST = NTE < 50
log(f"data dir: {DATA_DIR} | train {NTR} rows, {len(np.unique(gtr))} prompts | test {NTE} rows | CV folds {FOLD_IDS}")
if TINY_TEST and CFG["SKIP_HEAVY_IF_TINY_TEST"]:
    CFG["USE_LLM"] = CFG["USE_DEBERTA"] = False
    log("tiny test -> GPU components disabled (SKIP_HEAVY_IF_TINY_TEST)")

### Model discovery

In [ ]:
MODEL_SPECS = {
    # key: name patterns (regex on the lower-cased path), expected model_type, LDA recipe, defaults, T4 cost
    "q4b": dict(hf="Qwen/Qwen3-4B-Base", pats=[r"qwen3?[-_.]?4b[-_.]?base", r"(^|[^0-9.])4b[-_.]base"],
                mtype=("qwen3",), layers=list(range(2, 29, 2)), ncomp=1024, shrink=0.6, T=3.32, T_raw=3.20, bs=8,
                t4_min_per_1k=2.6),
    "q8b": dict(hf="Qwen/Qwen3-8B-Base", pats=[r"qwen3?[-_.]?8b[-_.]?base", r"(^|[^0-9.])8b[-_.]base"],
                mtype=("qwen3",), layers=list(range(2, 29, 2)), ncomp=1024, shrink=0.6, T=3.10, T_raw=3.19, bs=4,
                t4_min_per_1k=5.0),
    "q17": dict(hf="Qwen/Qwen3-1.7B-Base", pats=[r"qwen3?[-_.]?1\.7b[-_.]?base", r"(^|[^0-9.])1\.7b[-_.]base"],
                mtype=("qwen3",), layers=list(range(2, 23, 2)), ncomp=1024, shrink=0.6, T=3.38, T_raw=3.27, bs=16,
                t4_min_per_1k=1.2),
    "q06": dict(hf="Qwen/Qwen3-0.6B-Base", pats=[r"qwen3?[-_.]?0\.6b[-_.]?base", r"(^|[^0-9.])0\.6b[-_.]base"],
                mtype=("qwen3",), layers=list(range(2, 23, 2)), ncomp=512, shrink=0.5, T=2.49, T_raw=2.39, bs=16,
                t4_min_per_1k=0.6),
    "smol17": dict(hf="HuggingFaceTB/SmolLM2-1.7B", pats=[r"smollm2[-_.]?1\.7b"],
                   mtype=("llama",), layers=list(range(2, 25, 2)), ncomp=1024, shrink=0.6, T=3.38, T_raw=2.98,
                   bs=16, t4_min_per_1k=0.9),
    "olmo1b": dict(hf="allenai/OLMo-2-0425-1B", pats=[r"olmo[-_.]?2[-_.]?0425[-_.]?1b", r"olmo[-_.]?2[-_.]?1b"],
                   mtype=("olmo2",), layers=list(range(1, 17)), ncomp=1024, shrink=0.6, T=3.41, T_raw=3.10,
                   bs=16, t4_min_per_1k=0.75),
    # Qwen2.5-3B-Instruct (Kaggle Models qwen-lm/qwen2.5, variation '3b-instruct'): a post-trained, different-
    # generation LM; pc-LDA 0.571 alone, adds diversity to the Qwen3 block. 'instruct' is allowed for this one only.
    "q25i3": dict(hf="Qwen/Qwen2.5-3B-Instruct", pats=[r"qwen2\.5[-_.]?3b[-_.]?instruct", r"(^|[^0-9.])3b[-_.]instruct"],
                  allow=("instruct",), mtype=("qwen2",), layers=list(range(2, 29, 2)), ncomp=1024, shrink=0.6,
                  T=3.15, T_raw=3.05, bs=8, t4_min_per_1k=3.0),
    "deberta": dict(hf="microsoft/deberta-v3-large", pats=[r"deberta[-_.]?v3[-_.]?large"], mtype=("deberta-v2",)),
}
NEG_PATS = [r"instruct", r"[-_.]it([-_./]|$)", r"chat", r"gguf", r"awq", r"gptq", r"fp8", r"int4", r"int8", r"bnb",
            r"4bit", r"8bit", r"sft", r"dpo", r"rlvr", r"embedding", r"onnx", r"openvino", r"/\.git"]


def _weight_files(d):
    try:
        fs = os.listdir(d)
    except Exception:
        return []
    return [f for f in fs if f.endswith(".safetensors") or (f.startswith("pytorch_model") and f.endswith(".bin"))]


def scan_model_dirs(roots, max_depth=9):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        root = os.path.abspath(root)
        for dp, dns, fns in os.walk(root):
            rel = os.path.relpath(dp, root)
            depth = 0 if rel == "." else rel.count(os.sep) + 1
            if "config.json" in fns:
                found.append(dp)
            dns[:] = [d for d in dns if not d.startswith(".") and d not in ("blobs", "refs", "onnx", "openvino")]
            if depth >= max_depth:
                dns[:] = []
    return found


def _model_roots():
    if ON_KAGGLE:
        return ["/kaggle/input"]
    roots = [r for r in os.environ.get("NTL_MODEL_ROOTS", "").split(os.pathsep) if r]
    roots += [os.path.expanduser("~/.cache/huggingface/hub"), os.environ.get("HF_HOME", "")]
    return [r for r in roots if r and os.path.isdir(r)]


MODEL_DIRS = scan_model_dirs(_model_roots())


def resolve_model(key):
    """Best local folder for MODEL_SPECS[key] (with config.json + weights + matching model_type), or None."""
    if CFG["MODEL_PATHS"].get(key):
        return CFG["MODEL_PATHS"][key]
    spec = MODEL_SPECS[key]
    cands = []
    for d in MODEL_DIRS:
        norm = d.replace("\\", "/").lower()
        # match per path component: the component that names the model (Kaggle Models variation '4b-base',
        # an HF-linked model 'microsoft/deberta-v3-large', HF cache 'models--qwen--qwen3-4b-base') must hit a pattern,
        # and the negative patterns are checked on that component only (never on the user name / notebook slug)
        hits = [c for c in norm.rstrip("/").split("/") if any(re.search(p, c) for p in spec["pats"])]
        neg = [n for n in NEG_PATS if n not in spec.get("allow", ())]
        if not hits or all(any(re.search(n, h) for n in neg) for h in hits):
            continue
        try:
            with open(os.path.join(d, "config.json"), encoding="utf-8") as f:
                mt = json.load(f).get("model_type", "")
        except Exception:
            continue
        if spec.get("mtype") and mt not in spec["mtype"]:
            continue
        if not _weight_files(d):
            continue
        cands.append((norm.count("/"), len(norm), d))
    return sorted(cands)[0][2] if cands else None


MODEL_PATHS = {k: resolve_model(k) for k in MODEL_SPECS}
log(f"scanned {len(MODEL_DIRS)} model folders")
_WANTED = set(CFG["LLM_MODELS"]) if CFG["USE_LLM"] else set()
if CFG["USE_DEBERTA"]:
    _WANTED.add("deberta")
for k, p in MODEL_PATHS.items():
    if k in _WANTED:
        print(f"  {k:8s} -> {p if p else '!!! NOT FOUND (component will be skipped) -> attach ' + MODEL_SPECS[k]['hf']}")
    else:
        print(f"  {k:8s} -> (optional, not enabled){'  [found: ' + p + ']' if p else ''}")

## 2. Exploratory data analysis

What we want to learn before modelling:
1. **Structure**: how prompts and models are laid out. This decides the validation scheme and the post-processing.
2. **Length**: is passage length a shortcut? (The host says it should not be.)
3. **Typography fingerprints**: dashes, quotes, apostrophes and punctuation rates per model.
4. **Lexical fingerprints**: the words and openings each model over-uses.
5. **Model similarity**: which models are siblings. These are the hard confusions.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("whitegrid")
    HAS_PLOT = True
except Exception as e:  # noqa
    HAS_PLOT = False
    print("plotting unavailable:", e)

with section("EDA: structure"):
    if CFG["RUN_EDA"]:
        print(train.shape, test.shape)
        display(train.head(3))
        print('\nPassages per model:')
        print(train['model'].value_counts().sort_index().to_string())
        pm = train.groupby('prompt_id')['model'].agg(['count', 'nunique'])
        print(f"\n{train['prompt_id'].nunique()} prompts; passages per prompt: min={pm['count'].min()} max={pm['count'].max()}; "
              f"prompts where a model appears twice: {(pm['count'] != pm['nunique']).sum()}")

**Key structural fact:** every prompt was answered by every model **exactly once**. Test prompts are unseen, so
validation must be **GroupKFold by `prompt_id`**. On test we can (a) recover the prompt groups by topic clustering
and (b) enforce "each model at most once per prompt" with an assignment solver. This is the single biggest lever in
this competition.

In [ ]:
with section("EDA: length"):
    if CFG["RUN_EDA"] and HAS_PLOT:
        eda = train.copy()
        eda['n_words'] = eda['text'].str.split().str.len()
        eda['rel_len'] = eda['n_words'] / eda.groupby('prompt_id')['n_words'].transform('mean')
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))
        sns.boxplot(data=eda, y='model', x='n_words', order=LABELS, ax=axes[0], color='#8fb3d9')
        axes[0].set_title('Words per passage by model')
        sns.boxplot(data=eda, y='model', x='rel_len', order=LABELS, ax=axes[1], color='#d9b38f')
        axes[1].set_title('Length relative to the prompt mean')
        axes[1].set_yticklabels([])
        axes[1].set_ylabel('')
        plt.tight_layout()
        plt.show()
        print('within-prompt std of length:', round(eda.groupby('prompt_id')['n_words'].std().mean(), 1),
              '| overall std:', round(eda['n_words'].std(), 1))
        if len(test) > 0:
            print('test words:', test['text'].str.split().str.len().describe().round(1).to_dict())

Length is nearly useless. It varies as much inside a prompt as across the whole dataset, because the generator
randomised the target length. Only `gemini-3.8-flash` and `kimi-k3` are a little shorter on average.

In [ ]:
with section("EDA: typography"):
    if CFG["RUN_EDA"] and HAS_PLOT:
        TYPO = {
            'em dash —': '—', 'en dash –': '–', "curly ’": '’', "straight '": "'",
            'curly “': '“', 'straight "': '"', 'ellipsis …': '…', 'semicolon ;': ';',
            'colon :': ':', 'parenthesis (': '(', 'hyphen -': '-', 'digit': None, 'comma ,': ',',
        }
        nw = train['text'].str.split().str.len()
        rows = {}
        for name, ch in TYPO.items():
            cnt = train['text'].str.count(r'\d') if ch is None else train['text'].str.count(re.escape(ch))
            rows[name] = (cnt / nw * 100).groupby(train['model']).mean()
        typo = pd.DataFrame(rows).loc[LABELS]
        plt.figure(figsize=(14, 8))
        sns.heatmap(typo, annot=True, fmt='.2f', cmap='mako_r', cbar_kws={'label': 'per 100 words'})
        plt.title('Typography fingerprint: occurrences per 100 words')
        plt.tight_layout()
        plt.show()
        use = pd.DataFrame({n: train['text'].str.contains(re.escape(c)).groupby(train['model']).mean()
                            for n, c in TYPO.items()
                            if c is not None and n in ('em dash —', "curly ’", 'curly “', 'straight "', 'semicolon ;')}).loc[LABELS]
        display((use * 100).round(0).astype(int))

Very strong, almost deterministic fingerprints:
* **Gemini never uses the em dash.** Every other vendor does, and Claude/GLM use it heavily.
* **Claude and GLM never use the curly apostrophe `’`.** GPT, Gemini-flash, Kimi and Qwen do.
* **GPT uses curly double quotes `“ ”`.** Claude and GLM use straight `"`.

So the vendor/family is often decidable from typography alone. The real difficulty is telling **siblings** apart:
gpt-5.6 luna/sol/terra, gemini flash 3.6/3.7/3.8, claude opus 4-6/4-8/5, and glm vs glm-flash. Character n-grams
must therefore be computed **without lowercasing or unicode normalisation**.

In [ ]:
with section("EDA: lexical fingerprints"):
    if CFG["RUN_EDA"]:
        from sklearn.feature_extraction.text import CountVectorizer
        cv_words = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'’-]+\b", min_df=5)
        Xw_eda = cv_words.fit_transform(train['text'])
        vocab = np.array(cv_words.get_feature_names_out())
        Yd = pd.get_dummies(train['model'])[LABELS].values.astype(float)
        counts = np.asarray((Xw_eda.T @ Yd).T)                               # (18, V)
        alpha = 0.01 * counts.sum(0) + 1e-3                                  # informative Dirichlet prior
        tot = counts.sum(1, keepdims=True)
        rest = counts.sum(0, keepdims=True) - counts
        a0 = alpha.sum()
        lo = (np.log((counts + alpha) / (tot + a0 - counts - alpha))
              - np.log((rest + alpha) / (tot.sum() - tot + a0 - rest - alpha)))
        z = lo / np.sqrt(1 / (counts + alpha) + 1 / (rest + alpha))
        top_words = pd.DataFrame({m: vocab[np.argsort(-z[i])[:12]] for i, m in enumerate(LABELS)}).T
        top_words.columns = [f'#{i + 1}' for i in range(12)]
        print('Most over-used words per model (log-odds z-score with informative prior):')
        display(top_words)
        first2 = train['text'].str.split().str[:2].str.join(' ').str.lower()
        print('Most common openings per model:')
        display(pd.DataFrame({m: first2[train['model'] == m].value_counts().head(5).index.tolist() for m in LABELS}).T)

In [ ]:
with section("EDA: model similarity"):
    if CFG["RUN_EDA"] and HAS_PLOT:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from scipy.cluster.hierarchy import linkage as _linkage, dendrogram
        tf_char = TfidfVectorizer(analyzer='char', ngram_range=(1, 4), lowercase=False, sublinear_tf=True, min_df=3)
        Xc_eda = tf_char.fit_transform(train['text'])
        cent = np.vstack([np.asarray(Xc_eda[train['model'].values == m].mean(0)) for m in LABELS])
        cent = cent / np.linalg.norm(cent, axis=1, keepdims=True)
        fig, axes = plt.subplots(1, 2, figsize=(20, 7), gridspec_kw={'width_ratios': [1.2, 1]})
        sns.heatmap(pd.DataFrame(cent @ cent.T, index=LABELS, columns=LABELS), cmap='rocket_r', ax=axes[0])
        axes[0].set_title('Cosine similarity of model centroids (char 1-4 TF-IDF)')
        dendrogram(_linkage(cent, 'average', metric='cosine'), labels=LABELS, orientation='right', ax=axes[1])
        axes[1].set_title('Stylistic family tree')
        plt.tight_layout()
        plt.show()
        del Xc_eda, Xw_eda

In [ ]:
with section("EDA: example"):
    if CFG["RUN_EDA"]:
        _pid = train['prompt_id'].value_counts().index[0]
        print(f'Six answers to the same prompt ({_pid}):\n')
        for _, r in train[train['prompt_id'] == _pid].head(6).iterrows():
            print(f"[{r['model']}] {r['text'][:260]}...\n")

The six answers above share the topic words of the prompt but not the style. That is exactly why we
**prompt-centre** every feature: subtracting the group mean removes the shared topic and keeps the
model-specific residual.

## 3. Validation scheme and prompt-group recovery

**Validation.** Test prompts never appear in train, so every OOF number comes from **GroupKFold by `prompt_id`**:
5 folds of 30 prompts (540 passages) each. A random KFold would leak each prompt's topic between folds.

**Test groups.** The test set is built like train: about 222 prompts, each answered by up to 18 models. We recover
the groups without labels:
* The similarity is 0.5 x word TF-IDF + 0.5 x char_wb 4-5 TF-IDF cosine. The word TF-IDF drops stop words and uses
  prefix-6 stemming. Both are fitted on the test texts themselves.
* Average linkage, cut at distance 0.94.
* **Any cluster larger than 18 is split again** by its own sub-dendrogram. This split step is essential: without it
  accuracy drops by 0.05.

On train this places 96-98% of passages in exactly their true group. Decoding with predicted clusters instead of
the true groups costs only 0.001-0.003 balanced accuracy. This holds with 0-60% per-model dropout and with
50/100/150 prompts.

The cell below is the group toolbox (clustering, centring, Sinkhorn + pairwise decoding). It is inlined from
the project's `postproc.py`.

In [ ]:
from scipy.optimize import linear_sum_assignment, minimize_scalar
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics import (adjusted_rand_score, homogeneity_score, completeness_score, balanced_accuracy_score,
                             confusion_matrix)

_STOP = frozenset(ENGLISH_STOP_WORDS)


def softmax(z, T=1.0):
    z = np.asarray(z, np.float64) / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def bacc(y, p):
    p = np.asarray(p)
    return balanced_accuracy_score(y, p.argmax(1) if p.ndim == 2 else p)


# ---------------------------------------------------------------- clustering
def _word_stem_analyzer(t, stem_len=6):
    return [w[:stem_len] for w in re.findall(r"[a-z][a-z]+", t.lower()) if w not in _STOP]


def tfidf_features(texts, kind="word", max_df=0.3, stem_len=6):
    if kind == "word":
        v = TfidfVectorizer(analyzer=lambda t: _word_stem_analyzer(t, stem_len), sublinear_tf=True, min_df=1, max_df=max_df)
    else:
        v = TfidfVectorizer(analyzer="char_wb", ngram_range=(4, 5), lowercase=True, sublinear_tf=True, min_df=2, max_df=max_df)
    return normalize(v.fit_transform(list(texts)))


def text_similarity(texts, w_word=0.5, w_char=0.5, max_df=0.3):
    parts, ws = [], []
    if w_word > 0:
        X = tfidf_features(texts, "word", max_df=max_df); parts.append((X @ X.T).toarray()); ws.append(w_word)
    if w_char > 0:
        X = tfidf_features(texts, "char", max_df=max_df); parts.append((X @ X.T).toarray()); ws.append(w_char)
    ws = np.asarray(ws) / np.sum(ws)
    S = sum(w * p for w, p in zip(ws, parts))
    S = (S + S.T) / 2
    np.fill_diagonal(S, 1.0)
    return S


def _linkage_from_sim(S, method="average"):
    D = np.clip((1.0 - S + (1.0 - S).T) / 2, 0, None)
    np.fill_diagonal(D, 0.0)
    return linkage(squareform(D, checks=False), method=method)


def _split_oversize(S, lab, max_size, method):
    lab = lab.copy(); nxt = lab.max() + 1
    for _ in range(10):
        changed = False
        for c in np.unique(lab):
            idx = np.where(lab == c)[0]
            if len(idx) <= max_size:
                continue
            k = int(np.ceil(len(idx) / max_size))
            sub = fcluster(_linkage_from_sim(S[np.ix_(idx, idx)], method), t=k, criterion="maxclust") - 1
            if len(np.unique(sub)) <= 1:
                continue
            for s in np.unique(sub)[1:]:
                lab[idx[sub == s]] = nxt; nxt += 1
            changed = True
        if not changed:
            break
    return lab


def cluster_prompts(texts, threshold=0.94, method="average", w_word=0.5, w_char=0.5, max_size=NC, max_df=0.3):
    S = text_similarity(texts, w_word=w_word, w_char=w_char, max_df=max_df)
    if S.shape[0] == 1:
        return np.zeros(1, int)
    lab = fcluster(_linkage_from_sim(S, method), t=threshold, criterion="distance") - 1
    lab = _split_oversize(S, lab, max_size, method)
    return np.unique(lab, return_inverse=True)[1].astype(int)


def safe_cluster(texts):
    """cluster_prompts that never fails: tiny inputs (max_df pruning can empty the vocabulary) retry with max_df=1,
    and the last resort is one cluster per passage."""
    texts = list(texts); n = len(texts)
    if n <= 1:
        return np.zeros(n, int)
    # tiny inputs: max_df=0.3 empties the vocabulary -> topic-word channel only, no df pruning
    tries = [dict(CFG["CLUSTER"]), dict(CFG["CLUSTER"], w_word=1.0, w_char=0.0, max_df=1.0)]
    for kw in tries:
        try:
            return np.asarray(cluster_prompts(texts, **kw), int)
        except Exception as e:  # noqa
            log(f"cluster_prompts({kw}) failed on {n} texts: {e}")
    return np.arange(n)


def cluster_report(groups, true):
    groups, true = np.asarray(groups), np.asarray(true)
    gsets, tsets = {}, {}
    for i, (g, t) in enumerate(zip(groups, true)):
        gsets.setdefault(g, set()).add(i); tsets.setdefault(t, set()).add(i)
    exact = np.mean([gsets[g] == tsets[t] for g, t in zip(groups, true)])
    sizes = np.bincount(np.unique(groups, return_inverse=True)[1])
    return dict(ari=round(float(adjusted_rand_score(true, groups)), 4), hom=round(float(homogeneity_score(true, groups)), 4),
                comp=round(float(completeness_score(true, groups)), 4), exact=round(float(exact), 4),
                n_clusters=len(sizes), n_true=len(np.unique(true)), n_oversize=int((sizes > NC).sum()),
                n_singleton=int((sizes == 1).sum()))


def group_center(X, groups):
    """Subtract the per-group mean (prompt-centring). Works in float32."""
    X = np.asarray(X, dtype=np.float32).copy()
    groups = np.asarray(groups)
    for g in np.unique(groups):
        i = np.where(groups == g)[0]
        X[i] -= X[i].mean(0, keepdims=True)
    return X


# ---------------------------------------------------------------- assignment inside groups
def _hungarian_block(lp, cap=None):
    m, C = lp.shape
    cap = int(np.ceil(m / C)) if cap is None else cap
    r, c = linear_sum_assignment(-np.tile(lp, (1, cap)))
    out = np.empty(m, int); out[r] = c % C
    return out


def _col_scale(Q, col, cap, target):
    colsum = Q.sum(0)
    if col == "eq":
        return target / np.maximum(colsum, 1e-300)
    if isinstance(col, (float, int)) and not isinstance(col, bool):
        scale = np.power(target / np.maximum(colsum, 1e-300), float(col))
        return np.minimum(scale, cap / np.maximum(colsum * scale, 1e-300) * scale)
    return np.where(colsum > cap, cap / colsum, 1.0)


def _sinkhorn_block(P, n_iter=50, power=1.0, col="le"):
    """Approximate marginals of 'each class at most cap = max(1, m/C) times' (col='le'); damped balancing for a float col."""
    m, C = P.shape
    cap, target = max(1.0, m / C), m / C
    Q = np.power(np.clip(P, 1e-300, None), power)
    Q = Q / Q.sum(1, keepdims=True)
    for _ in range(n_iter):
        Q = Q * _col_scale(Q, col, cap, target)[None, :]
        Q = Q / Q.sum(1, keepdims=True)
    return Q


def _sinkhorn_logits(U, col="le", n_iter=30):
    m, C = U.shape
    cap, target = max(1.0, m / C), m / C
    Q = np.exp(U - U.max(1, keepdims=True))
    Q = Q / Q.sum(1, keepdims=True)
    for _ in range(n_iter):
        Q = Q * _col_scale(Q, col, cap, target)[None, :]
        Q = Q / Q.sum(1, keepdims=True)
    return Q


def fit_temperature(proba, y, eps=1e-12):
    """power = 1/T minimising the NLL of softmax(power * log proba)."""
    lp = np.log(np.asarray(proba, float) + eps); y = np.asarray(y)

    def nll(pw):
        z = lp * pw
        z = z - z.max(1, keepdims=True)
        z = z - np.log(np.exp(z).sum(1, keepdims=True))
        return -z[np.arange(len(y)), y].mean()
    return float(minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)


def fit_T_logits(Z, y, default=1.0):
    """temperature T minimising NLL of softmax(Z / T)."""
    try:
        y = np.asarray(y)
        def nll(t):
            P = softmax(Z / t)
            return -np.log(P[np.arange(len(y)), y] + 1e-12).mean()
        return float(minimize_scalar(nll, bounds=(0.05, 50), method="bounded").x)
    except Exception:
        return default


# ---------------------------------------------------------------- pairwise sibling-similarity term
_STYLO_CHARS = [",", ";", ":", "—", "–", "-", "“", "”", "‘", "’", "'", '"', "(", "!", "?", "…", "*", "\n"]


def stylo_matrix(texts):
    rows = []
    for t in texts:
        words = t.split(); nw = max(len(words), 1)
        sents = max(len(re.findall(r"[.!?]+(?:\s|$)", t)), 1)
        r = [t.count(c) / nw for c in _STYLO_CHARS]
        r += [np.mean([len(w) for w in words]) if words else 0.0, nw / sents, np.log(nw),
              sum(w[:1].isupper() for w in words) / nw, sum(ch.isdigit() for ch in t) / max(len(t), 1)]
        rows.append(r)
    X = np.asarray(rows, float)
    return (X - X.mean(0)) / (X.std(0) + 1e-9)


PAIR_CHANNELS = ("word13", "char35", "stylo", "open")


def pair_similarities(texts, channels=PAIR_CHANNELS):
    texts = list(texts); out = {}
    for ch in channels:
        if ch == "word13":
            X = normalize(TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True, min_df=1, lowercase=True).fit_transform(texts))
            out[ch] = (X @ X.T).toarray()
        elif ch == "char35":
            X = normalize(TfidfVectorizer(analyzer="char", ngram_range=(3, 5), sublinear_tf=True,
                                          min_df=2 if len(texts) > 2 else 1, lowercase=False).fit_transform(texts))
            out[ch] = (X @ X.T).toarray()
        elif ch == "stylo":
            X = normalize(stylo_matrix(texts)); out[ch] = X @ X.T
        elif ch == "open":
            X = normalize(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=1,
                                          lowercase=True).fit_transform([t[:120] for t in texts]))
            out[ch] = (X @ X.T).toarray()
    return out


def pair_blocks(sims, groups, min_size=3):
    """Within-group z-scored similarities -> {group: (idx, Z (K, m, m))}."""
    mats = list(sims.values()) if isinstance(sims, dict) else list(sims)
    groups = np.asarray(groups); out = {}
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]; m = len(idx)
        if m < min_size:
            continue
        off = ~np.eye(m, dtype=bool)
        Z = np.zeros((len(mats), m, m))
        for k, S in enumerate(mats):
            Sg = S[np.ix_(idx, idx)]; v = Sg[off]
            Zg = (Sg - v.mean()) / (v.std() + 1e-9); Zg[~off] = 0.0
            Z[k] = Zg
        out[g] = (idx, Z)
    return out


def fit_pair_model(blocks, y, ridge=1e-3, C=NC):
    """z_ij ~ N(mu[a, b], Sigma): per model pair mean of the K channels + pooled covariance (LDA-style)."""
    y = np.asarray(y)
    K = next(iter(blocks.values()))[1].shape[0]
    S1 = np.zeros((K, C, C)); N = np.zeros((C, C))
    for idx, Z in blocks.values():
        yy = y[idx]
        A = yy[:, None].repeat(len(idx), 1); B = yy[None, :].repeat(len(idx), 0); off = A != B
        for k in range(K):
            np.add.at(S1[k], (A[off], B[off]), Z[k][off])
        np.add.at(N, (A[off], B[off]), 1)
    mu = S1 / np.maximum(N, 1)[None]
    mu = (mu + mu.transpose(0, 2, 1)) / 2
    for k in range(K):
        np.fill_diagonal(mu[k], 0.0)
    R = []
    for idx, Z in blocks.values():
        yy = y[idx]; iu = np.triu_indices(len(idx), 1)
        a, b = yy[iu[0]], yy[iu[1]]; keep = a != b
        R.append((Z[:, iu[0], iu[1]] - mu[:, a, b])[:, keep])
    Sig = np.atleast_2d(np.cov(np.concatenate(R, 1))) + ridge * np.eye(K)
    W = np.linalg.inv(Sig)
    return dict(mu=mu, W=W, Cmat=np.einsum("kab,kl,lab->ab", mu, W, mu), K=K)


def _pair_meanfield(L, Z, model, lam, col="le", mf_iter=15, mf_damp=0.5, sk_iter=30):
    mu, W, Cm = model["mu"], model["W"], model["Cmat"]
    Zt = np.einsum("kl,lij->kij", W, Z)
    Q = _sinkhorn_logits(L, col, sk_iter)
    for _ in range(mf_iter):
        U = L.copy()
        for k in range(Zt.shape[0]):
            U += lam * (Zt[k] @ (Q @ mu[k]))
        QC = Q @ Cm
        U -= lam * 0.5 * (QC.sum(0, keepdims=True) - QC)
        Q = mf_damp * Q + (1 - mf_damp) * _sinkhorn_logits(U, col, sk_iter)
    return Q


def assign(proba, groups, method="sinkhorn", power=1.0, col="le", pair=None, pair_lam=0.5, mf_iter=15,
           mf_damp=0.5, final="argmax", min_group=2, n_iter=50, eps=1e-9, return_scores=False):
    """'argmax' | 'hungarian' | 'sinkhorn' (+ pairwise mean-field term when pair=(blocks, model) is given).
    Groups smaller than min_group keep the argmax."""
    proba = np.asarray(proba, float); proba = proba / proba.sum(1, keepdims=True)
    groups = np.asarray(groups)
    pred = proba.argmax(1); scores = proba.copy()
    if method == "argmax":
        return (pred, scores) if return_scores else pred
    lp = np.log(proba + eps)
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        if len(idx) < min_group:
            continue
        if method == "hungarian":
            pred[idx] = _hungarian_block(lp[idx]); scores[idx] = np.eye(proba.shape[1])[pred[idx]]
            continue
        blk = None if pair is None else pair[0].get(g)
        if blk is not None and pair_lam > 0:
            Q = _pair_meanfield(power * np.log(np.clip(proba[idx], 1e-300, None)), blk[1], pair[1], pair_lam, col,
                                mf_iter, mf_damp)
        else:
            Q = _sinkhorn_block(proba[idx], n_iter=n_iter, power=power, col=col)
        pred[idx] = _hungarian_block(np.log(Q + eps)) if final == "hungarian" else Q.argmax(1)
        scores[idx] = Q
    return (pred, scores) if return_scores else pred


def fit_pair_model_from_texts(texts, y, groups, channels=PAIR_CHANNELS):
    S = pair_similarities(texts, channels)
    return fit_pair_model(pair_blocks(S, groups), y), S


def decode_groups(proba, texts, groups, pair_model, dec_cfg, power):
    """Sinkhorn + pairwise decoding with graceful degradation: pair -> plain Sinkhorn -> Hungarian -> argmax."""
    kw = dict(method="sinkhorn", power=power, col=dec_cfg.get("col", "le"), pair_lam=dec_cfg.get("pair_lam", 0.5),
              mf_iter=dec_cfg.get("mf_iter", 15), mf_damp=dec_cfg.get("mf_damp", 0.5), final=dec_cfg.get("final", "argmax"))
    try:
        if pair_model is None or kw["pair_lam"] <= 0:
            raise RuntimeError("no pair model")
        S = pair_similarities(texts)
        return assign(proba, groups, pair=(pair_blocks(S, groups), pair_model), return_scores=True, **kw), "sinkhorn+pair"
    except Exception as e:  # noqa
        log(f"pairwise decoding unavailable ({e}); trying plain Sinkhorn")
    try:
        kw.pop("pair_lam", None)
        return assign(proba, groups, return_scores=True, **kw), "sinkhorn"
    except Exception as e:  # noqa
        log(f"Sinkhorn failed ({e}); trying Hungarian")
    try:
        return assign(proba, groups, method="hungarian", return_scores=True), "hungarian"
    except Exception as e:  # noqa
        log(f"Hungarian failed ({e}); argmax")
    return (np.asarray(proba).argmax(1), np.asarray(proba)), "argmax"

### Recover the test groups (and, for in-notebook CV, the groups of every validation fold)

For the OOF evaluation we cluster each validation fold **on its own**, exactly as we cluster the test. This is
more honest than centring with the true prompt ids.

In [ ]:
test_groups = np.zeros(NTE, int)
with section("clustering"):
    test_groups = safe_cluster(te_texts)
    sizes = np.bincount(test_groups)
    TEST_CLU_SIZE = sizes[test_groups]
    log(f"test: {NTE} passages -> {len(sizes)} clusters; size distribution: "
        f"{pd.Series(sizes).value_counts().sort_index().to_dict()}")
    if PID_HOLDOUT is not None:
        log(f"holdout clustering vs true prompts: {cluster_report(test_groups, PID_HOLDOUT)}")
TEST_CLU_SIZE = np.bincount(test_groups)[test_groups]

oof_groups = None
with section("train clustering check + OOF fold clusters"):
    rep = cluster_report(safe_cluster(tr_texts), pid_tr)
    log(f"clustering the whole train ({NTR} texts, {len(np.unique(pid_tr))} prompts): {rep}")
    if CFG["RUN_CV"]:
        oof_groups = np.zeros(NTR, int); off = 0
        for f in FOLD_IDS:
            va = np.where(folds == f)[0]
            g = safe_cluster([tr_texts[i] for i in va])
            oof_groups[va] = g + off; off += g.max() + 1
        log(f"OOF fold clusters vs true prompts: {cluster_report(oof_groups, pid_tr)}")


def pc_weight(sizes):
    """Weight of the prompt-centred blend for rows in clusters of size m: clip((m - 1) / SMALL_DENOM, 0, 1).
    A singleton centres to the zero vector, so it relies entirely on the uncentred twins."""
    return np.clip((np.asarray(sizes, float) - 1) / float(CFG["SMALL_DENOM"]), 0, 1)


ALPHA_TE = pc_weight(TEST_CLU_SIZE)
log(f"test rows using the uncentred fallback (cluster size < {CFG['SMALL_DENOM'] + 1}): "
    f"{int((ALPHA_TE < 1).sum())} / {NTE}")

## 4. Component A: TF-IDF + stylometry, prompt-centred (CPU)

**Why:** typography and phrasing fingerprints (em dashes, curly quotes, function-word rates, sentence openers) are
cheap, topic-light signals. They are complementary to LM hidden states.

**Recipe** (CV argmax 0.473 with clustered centring; 0.582 after Sinkhorn + pair decoding):
* Features:
  * char 1-5 TF-IDF, case-sensitive with no unicode normalisation, at most 300k features;
  * word 1-2 TF-IDF where punctuation counts as tokens;
  * 442 stylometric features (punctuation rates, dash spacing, word/sentence length statistics, TTR/Yule,
    sentence openers, about 330 function-word rates, regex constructions), signed-log, standardised, then x0.5.
* **Centring through the Gram matrix.** For a linear model, prompt-centring the features equals
  `Kc = K - GK - KG + GKG`, where G block-averages rows by group (train by `prompt_id`, test by cluster). We
  eigen-decompose the train block and project the test rows exactly. This is exact and cheap.
* Multinomial LR (C=10) on the eigen-features, plus 0.2 x a multinomial NB on char 1-5 counts. The NB scores are
  length-normalised log-likelihoods, group-centred, and passed through a softmax with T=0.0057.
* An **uncentred twin** (same Gram, no centring; CV 0.397) is the fallback for passages in tiny clusters, where
  centring is meaningless: a singleton centres to the zero vector.

In [ ]:
FW = sorted(ENGLISH_STOP_WORDS | {"yet", "though", "although", "whereas", "while", "thus", "hence", "therefore",
                                   "indeed", "instead", "rather", "simply", "merely", "perhaps", "often", "also",
                                   "just", "even", "still", "only", "really", "quite", "truly", "deeply", "can",
                                   "may", "might", "must", "shall", "will", "would", "could", "should", "it's",
                                   "isn't", "don't", "doesn't", "can't", "won't", "that's", "what's", "let's"})
FWSET = set(FW)
SPECIAL = ["—", "–", "-", "’", "'", "‘", '"', "“", "”", ";", ":", "(", ")", "?", "!",
           "…", "*", ",", ".", "/", "[", "]", "−", "%", "&", "_", "#", "`"]
TOK_RE = re.compile(r"\w+(?:[’']\w+)*|[^\w\s]")
SENT_RE = re.compile(r"(?<=[.!?])[\"”’)]?\s+(?=[A-Z\"“(‘'])")
OPENERS = ("the", "this", "it", "in", "but", "and", "yet", "a", "we", "that", "these", "such", "when", "if",
           "for", "what", "each", "every", "no", "as", "by", "there", "they", "ultimately", "instead", "here",
           "so", "thus", "rather", "at", "to", "one", "even", "still", "consider", "imagine")
PATS = {
    "not_but": r"\bnot\b[^.;]{0,40}\bbut\b", "not_just": r"\bnot (just|merely|only|simply)\b",
    "isnt": r"\b(isn't|aren't|wasn't)\b", "rather_than": r"\brather than\b", "in_other": r"\bin other words\b",
    "ultimately": r"\bultimately\b", "colon_list": r":\s+\w+,", "x_y_and_z": r"\w+, \w+, and \w+",
    "x_y_z_no_and": r"\w+, \w+, \w+[.;]", "oxford": r", and\b", "comma_but": r", but\b", "semicolon_but": r"; but\b",
    "which": r"\bwhich\b", "ing_comma": r"\w+ing,", "by_ing": r"\bby \w+ing\b", "both_and": r"\bboth\b.*\band\b",
    "whether": r"\bwhether\b", "thus": r"\bthus\b", "itself": r"\b\w+self\b", "we_us": r"\b(we|us|our)\b",
    "you": r"\b(you|your)\b", "i_me": r"\b(i|me|my)\b", "quote_words": r"[\"“]\w+[\"”]",
    "paren": r"\([^)]*\)", "eg_ie": r"\b(e\.g\.|i\.e\.)", "number_word": r"\b\d+\b",
}


def norm_apos(t):
    return t.replace("’", "'").replace("‘", "'")


def sentences(t):
    s = [x for x in SENT_RE.split(t.strip()) if x.strip()]
    return s if s else [t]


def stylo_one(t):
    f = {}
    nchar = len(t)
    toks = TOK_RE.findall(t)
    words = [w for w in toks if re.match(r"\w", w)]
    nw = max(len(words), 1)
    f["nchar"] = np.log1p(nchar); f["nword"] = np.log1p(nw)
    for ch in SPECIAL:
        c = t.count(ch)
        f[f"cnt_{ord(ch)}"] = c / nw * 100; f[f"has_{ord(ch)}"] = float(c > 0)
    f["emdash_spaced"] = len(re.findall(r" — ", t)) / nw * 100
    f["emdash_tight"] = len(re.findall(r"\w—\w", t)) / nw * 100
    f["endash_spaced"] = len(re.findall(r" – ", t)) / nw * 100
    f["hyphen_spaced"] = len(re.findall(r" - ", t)) / nw * 100
    f["double_hyphen"] = t.count("--") / nw * 100
    f["hyph_compound"] = len(re.findall(r"\w-\w", t)) / nw * 100
    f["digits"] = sum(c.isdigit() for c in t) / max(nchar, 1) * 100
    f["upper"] = sum(c.isupper() for c in t) / max(nchar, 1) * 100
    f["nonascii"] = sum(ord(c) > 127 for c in t) / max(nchar, 1) * 100
    f["spaces_double"] = t.count("  "); f["newlines"] = t.count("\n")
    f["ends_period"] = float(t.rstrip().endswith(".")); f["ends_q"] = float(t.rstrip().endswith("?"))
    f["ends_excl"] = float(t.rstrip().endswith("!"))
    f["ends_quote"] = float(t.rstrip()[-1:] in "\"”'’"); f["starts_quote"] = float(t.lstrip()[:1] in "\"“'‘")
    wl = np.array([len(w) for w in words]) if words else np.array([0])
    f["wl_mean"] = wl.mean(); f["wl_std"] = wl.std()
    for k in (3, 6, 8, 10, 12):
        f[f"wl_ge{k}"] = (wl >= k).mean()
    lw = [norm_apos(w.lower()) for w in words]
    cnt = {}
    for w in lw:
        cnt[w] = cnt.get(w, 0) + 1
    f["ttr"] = len(cnt) / nw
    f["hapax"] = sum(1 for v in cnt.values() if v == 1) / nw
    f["yule"] = (sum(v * v for v in cnt.values()) - nw) / (nw * nw) * 1e4
    f["fw_ratio"] = sum(1 for w in lw if w in FWSET) / nw
    f["contractions"] = sum(1 for w in lw if "'" in w) / nw * 100
    f["cap_mid"] = sum(1 for i, w in enumerate(words) if i > 0 and w[0].isupper()) / nw * 100
    ss = sentences(t)
    sl = np.array([len(TOK_RE.findall(s)) for s in ss])
    f["n_sent"] = len(ss); f["sl_mean"] = sl.mean(); f["sl_std"] = sl.std(); f["sl_max"] = sl.max(); f["sl_min"] = sl.min()
    f["sl_cv"] = sl.std() / max(sl.mean(), 1); f["sl_first"] = sl[0]; f["sl_last"] = sl[-1]
    f["sl_first_rel"] = sl[0] / max(sl.mean(), 1); f["sl_last_rel"] = sl[-1] / max(sl.mean(), 1)
    f["comma_per_sent"] = t.count(",") / len(ss)
    f["short_sent"] = (sl <= 8).mean(); f["long_sent"] = (sl >= 35).mean()
    first = []
    for s in ss:
        tk = TOK_RE.findall(s)
        first.append(norm_apos(tk[0].lower()) if tk else "")
    for w in OPENERS:
        f[f"open_{w}"] = sum(1 for x in first if x == w) / len(ss)
    f["open_ing"] = sum(1 for x in first if x.endswith("ing")) / len(ss)
    f["open_ly"] = sum(1 for x in first if x.endswith("ly")) / len(ss)
    for w in FW:
        f[f"fw_{w}"] = cnt.get(w, 0) / nw * 100
    low = norm_apos(t.lower())
    for k, p in PATS.items():
        f[f"pat_{k}"] = len(re.findall(p, low)) / nw * 100
    return f


def stylo_frame(texts):
    return pd.DataFrame([stylo_one(t) for t in texts]).astype(np.float32)


def group_avg_matrix(groups):
    import scipy.sparse as sp
    codes, inv = np.unique(np.asarray(groups), return_inverse=True)
    n = len(inv); cnt = np.bincount(inv)
    G = sp.csr_matrix((1.0 / cnt[inv], (np.arange(n), inv)), shape=(n, len(codes)))
    M = sp.csr_matrix((np.ones(n), (inv, np.arange(n))), shape=(len(codes), n))
    return (G @ M).tocsr()


def center_K(K, groups):
    G = group_avg_matrix(groups)
    GK = np.asarray(G @ K)
    GKG = np.asarray(G @ GK.T)
    return K - GK - GK.T + GKG


def _expand(P, classes, C=NC):
    out = np.full((P.shape[0], C), 1e-9)
    out[:, np.asarray(classes, int)] = P
    return out / out.sum(1, keepdims=True)


def tf_build(all_texts, ntr):
    """Gram K = K_char + K_word + 0.5 K_stylo over train+test (vectorisers fitted on all texts, unsupervised) and the
    char 1-5 count matrix for the NB."""
    t = time.time()
    vc = TfidfVectorizer(analyzer="char", ngram_range=(1, 5), sublinear_tf=True, min_df=2, lowercase=False,
                         max_features=300000, dtype=np.float32)
    vw = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, min_df=2, lowercase=False,
                         token_pattern=r"(?u)\b\w+\b|[^\w\s]", dtype=np.float32)
    Xc = vc.fit_transform(all_texts)
    K = (Xc @ Xc.T).toarray().astype(np.float64); del Xc
    Xw = vw.fit_transform(all_texts)
    K += (Xw @ Xw.T).toarray(); del Xw
    S = stylo_frame(all_texts)
    S = S.loc[:, S.iloc[:ntr].std() > 0].values.astype(np.float64)
    S = np.sign(S) * np.log1p(np.abs(S))
    d = S.shape[1]
    S = np.clip(StandardScaler().fit_transform(S), -5, 5) / np.sqrt(d)
    K += 0.5 * (S @ S.T)
    Cn = CountVectorizer(analyzer="char", ngram_range=(1, 5), lowercase=False, min_df=2).fit_transform(all_texts)
    Cn = Cn.astype(np.float64).tocsr()
    log(f"TF features: Gram {K.shape}, stylo {d} dims, NB vocab {Cn.shape[1]} ({time.time() - t:.0f}s)")
    return K, Cn


def tf_lr(K, tr_idx, te_idx, ytr, gtr_, gte_, center=True, C=10.0):
    """Multinomial LR on (centred) Gram eigen-features of the train block; exact projection of the test rows."""
    sub = np.concatenate([tr_idx, te_idx])
    Ks = K[np.ix_(sub, sub)]
    if center:
        gt = np.unique(gtr_, return_inverse=True)[1]
        ge = np.unique(gte_, return_inverse=True)[1] + gt.max() + 1
        Ks = center_K(Ks, np.concatenate([gt, ge]))
    n = len(tr_idx)
    w, V = np.linalg.eigh(Ks[:n, :n])
    keep = w > 1e-8 * w.max()
    U, s = V[:, keep], np.sqrt(w[keep])
    lr = LogisticRegression(C=C, max_iter=3000, tol=1e-5).fit(U * s, ytr)
    return _expand(lr.predict_proba(Ks[n:, :n] @ U / s), lr.classes_)


def tf_nb(Cn, tr_idx, te_idx, ytr, gtr_, gte_, T=0.0057, alpha=10.0):
    nb = MultinomialNB(alpha=alpha).fit(Cn[tr_idx], ytr)

    def score(idx):
        return np.asarray(Cn[idx] @ nb.feature_log_prob_.T) / np.asarray(Cn[idx].sum(1)).clip(1)
    Z = group_center(score(te_idx), gte_) - group_center(score(tr_idx), gtr_).mean(0)
    return _expand(softmax(Z, T), nb.classes_)


def tf_unary(p_lr, p_nb, w_nb=0.2):
    return softmax((1 - w_nb) * np.log(np.clip(p_lr, 1e-9, None)) + w_nb * np.log(np.clip(p_nb, 1e-9, None)))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

COMP = {}      # name -> dict(test=centred probs, test_raw=uncentred twin, oof=OOF probs or None)
with section("A: TF-IDF + stylometry"):
    if CFG["USE_TF"]:
        K_tf, Cn_tf = tf_build(tr_texts + te_texts, NTR)
        tr_all, te_all = np.arange(NTR), np.arange(NTR, NTR + NTE)
        p_lr = tf_lr(K_tf, tr_all, te_all, y_tr, gtr, test_groups, center=True)
        p_nb = tf_nb(Cn_tf, tr_all, te_all, y_tr, gtr, test_groups)
        comp = dict(test=tf_unary(p_lr, p_nb), test_raw=None, oof=None)
        if (ALPHA_TE < 1).any():
            comp["test_raw"] = tf_lr(K_tf, tr_all, te_all, y_tr, gtr, test_groups, center=False)
        log("A: test predictions done")
        COMP["tf"] = comp
        if CFG["RUN_CV"] and oof_groups is not None:
            oof = np.zeros((NTR, NC))
            for f in FOLD_IDS:
                tr_i, va_i = np.where(folds != f)[0], np.where(folds == f)[0]
                ol = tf_lr(K_tf, tr_i, va_i, y_tr[tr_i], gtr[tr_i], oof_groups[va_i], center=True)
                on = tf_nb(Cn_tf, tr_i, va_i, y_tr[tr_i], gtr[tr_i], oof_groups[va_i])
                oof[va_i] = tf_unary(ol, on)
                log(f"  A fold {f}: bacc {bacc(y_tr[va_i], oof[va_i]):.4f}")
            comp["oof"] = oof
            log(f"A: in-notebook OOF bacc (val centred by predicted clusters) = {bacc(y_tr, oof):.4f}")
        del K_tf, Cn_tf
        gc.collect()

## 5. Component B: frozen decoder-LM hidden states, prompt-centred LDA

**Why:** a base LM's hidden states encode *how* a passage is written (token choice, rhythm, phrasing) far better than
n-grams. No fine-tuning is needed. LoRA on 0.6B did not beat frozen features, and the frozen 4B beat everything
we fine-tuned.

**Recipe** (each detail was ablated):
* Prepend `<|endoftext|>` as an attention-sink/BOS token and run fp16 inference (T4-safe; all features are finite).
* Mean-pool hidden states of about 14 layers (Qwen3-4B: layers 2, 4, ..., 28). **Position 0 (the sink) is
  excluded.** Concatenate the layers. Early and middle layers carry the most style; the last layers are the worst.
* **Prompt-centring**: train by `prompt_id`, test by predicted cluster. Raw features score 0.40-0.43 and centred
  ones 0.56-0.60.
* StandardScaler, then PCA(1024, `random_state=0`), then LDA(`lsqr`, shrinkage 0.6), then softmax(decision / T)
  with T about 3.3. Shrinkage LDA beats LR by about 2 points.
* Qwen3-4B: **hierarchical LDA**. A global 18-way LDA gives each family its mass; a per-family sub-LDA
  (PCA512, T=2.5) re-splits that mass among siblings (0.601 vs 0.596).
* An **uncentred twin LDA** is the fallback for passages in tiny clusters.

Honest CV (argmax / decoded alone): 4B-hier 0.598 / 0.707, 8B 0.584 / 0.693, 1.7B 0.575 / 0.697,
0.6B 0.563 / 0.681, SmolLM2 0.567 / 0.691, OLMo 0.575 / 0.681. The LMs are largely redundant, but together they
act as one diversity block: dropping several at once costs 0.004-0.009. The 8B needs 2xT4 (`device_map='auto'`)
and is skipped automatically on a single 16 GB GPU. The models are extracted one at a time, train and test in one
pass, and GPU memory is freed between them.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


def _weights_gb(path):
    tot = 0
    for f in _weight_files(path):
        try:
            tot += os.path.getsize(os.path.join(path, f))
        except Exception:
            pass
    try:
        with open(os.path.join(path, "config.json"), encoding="utf-8") as fh:
            cj = json.load(fh)
        dt = str(cj.get("torch_dtype") or cj.get("dtype") or "")
    except Exception:
        dt = ""
    gb = tot / 2**30
    return gb / 2 if "float32" in dt else gb


def from_pretrained_dtype(cls, path, dtype, **kw):
    """transformers compatibility: newer versions take dtype=, older ones torch_dtype= (an unknown kwarg would be
    silently swallowed into the config, so the dtype is verified afterwards)."""
    kw.setdefault("trust_remote_code", False)
    kw.setdefault("low_cpu_mem_usage", True)
    other = "torch_dtype" if DTYPE_KW == "dtype" else "dtype"
    try:
        m = cls.from_pretrained(path, **{DTYPE_KW: dtype}, **kw)
    except TypeError:
        m = cls.from_pretrained(path, **{other: dtype}, **kw)
    if dtype is not None:
        p = next(m.parameters())
        if p.dtype != dtype:
            log(f"  weights came as {p.dtype}; casting to {dtype}")
            m = m.to(dtype)
    return m


def extract_mean_feats(path, texts, layers, bs=8, maxlen=400, on_gpu=True):
    """Mean-pool (excluding the prepended sink token) of the given hidden-state layers -> float16 (n, len(layers)*H)."""
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=False)
    bos = tok.bos_token_id
    if bos is None:
        bos = tok.convert_tokens_to_ids("<|endoftext|>")
    if bos is None or (tok.unk_token_id is not None and bos == tok.unk_token_id):
        bos = tok.eos_token_id
    ids = [[bos] + tok(t, add_special_tokens=False)["input_ids"][:maxlen - 1] for t in texts]
    kw = {}
    gb = _weights_gb(path)
    multi = on_gpu and N_GPU > 1 and gb > 0.7 * GPU_MEM[0]
    if multi:
        kw["device_map"] = "auto"
        kw["max_memory"] = {i: f"{int(GPU_MEM[i] * 0.85)}GiB" for i in range(N_GPU)}
    model = from_pretrained_dtype(AutoModel, path, torch.float16 if on_gpu else torch.float32, **kw)
    if on_gpu and not multi:
        model = model.to("cuda")
    model.eval()
    dev = model.get_input_embeddings().weight.device
    H = model.config.hidden_size
    nl = model.config.num_hidden_layers
    layers = [l for l in layers if l <= nl]
    out = np.zeros((len(texts), len(layers) * H), dtype=np.float16)
    order = np.argsort([len(x) for x in ids])[::-1]
    t = time.time()
    hs = feats = None
    with torch.inference_mode():
        for b in range(0, len(order), bs):
            idx = order[b:b + bs]
            L = max(len(ids[i]) for i in idx)
            inp = torch.full((len(idx), L), bos, dtype=torch.long)
            att = torch.zeros((len(idx), L), dtype=torch.long)
            for j, i in enumerate(idx):
                inp[j, :len(ids[i])] = torch.tensor(ids[i]); att[j, :len(ids[i])] = 1
            inp, att = inp.to(dev), att.to(dev)
            hs = model(input_ids=inp, attention_mask=att, output_hidden_states=True).hidden_states
            m = att.float(); m[:, 0] = 0
            m = m / m.sum(1, keepdim=True).clamp(min=1)
            feats = [torch.einsum("bth,bt->bh", hs[l].to(m.device).float(), m) for l in layers]
            out[idx] = torch.cat(feats, 1).cpu().numpy().astype(np.float16)
            if b == 0 or (b // bs) % 200 == 0:
                log(f"    {b + len(idx)}/{len(order)} rows ({time.time() - t:.0f}s)")
    del model, hs, feats
    free_gpu()
    bad = ~np.isfinite(out)
    if bad.any():
        log(f"  WARNING: {int(bad.sum())} non-finite feature values -> 0")
        out[bad] = 0
    return out


def lda_decisions(Xtr, ytr, X_list, ncomp, shrink):
    """StandardScaler -> PCA -> shrinkage LDA fitted on Xtr; decision_function for every matrix in X_list.
    Columns follow sorted(unique(ytr)) (== class ids when all classes are present)."""
    Xtr = np.asarray(Xtr, np.float32)                       # never fit the scaler on float16
    sc = StandardScaler().fit(Xtr)
    Ztr = sc.transform(Xtr)
    nc = int(min(ncomp, Ztr.shape[0] - 1, Ztr.shape[1]))
    p = PCA(nc, random_state=0).fit(Ztr)
    m = LDA(solver="lsqr", shrinkage=shrink).fit(p.transform(Ztr), ytr)
    del Ztr
    return [m.decision_function(p.transform(sc.transform(np.asarray(X, np.float32)))) for X in X_list], m.classes_


def lda_full(Xtr, ytr, X_list, ncomp, shrink):
    Z, cls = lda_decisions(Xtr, ytr, X_list, ncomp, shrink)
    out = []
    for z in Z:
        full = np.full((z.shape[0], NC), -1e4); full[:, cls] = z; out.append(full)
    return out


def hier_decisions(Xtr, ytr, X_list, ncomp, shrink, sub_ncomp=512, sub_shrink=0.6):
    """Global 18-way LDA + per-family sub-LDAs (fit only on that family's rows)."""
    Zg = lda_full(Xtr, ytr, X_list, ncomp, shrink)
    Zs = []
    for fi in FAMI:
        mk = np.isin(ytr, fi)
        Zs.append(lda_decisions(Xtr[mk], ytr[mk], X_list, sub_ncomp, sub_shrink)[0])
    return Zg, Zs   # Zg[j]: (n_j, 18); Zs[k][j]: (n_j, |fam k|)


def hier_proba(Zg, Zs, Tg=3.3, Ts=2.5):
    Pg = softmax(Zg / Tg)
    P = np.zeros_like(Pg)
    for k, fi in enumerate(FAMI):
        P[:, fi] = Pg[:, fi].sum(1, keepdims=True) * softmax(Zs[k] / Ts)
    return P


def run_llm_component(key):
    spec = MODEL_SPECS[key]
    path = MODEL_PATHS.get(key)
    if not path:
        log(f"B[{key}]: model not found -> skipped"); return None
    on_gpu = HAS_CUDA
    if not on_gpu and not (CFG["ALLOW_CPU_LLM"] and key == "q06"):
        log(f"B[{key}]: no GPU -> skipped (set ALLOW_CPU_LLM for q06 on CPU)"); return None
    if on_gpu and N_GPU == 1 and _weights_gb(path) > 0.85 * GPU_MEM[0]:
        log(f"B[{key}]: {_weights_gb(path):.1f} GB of weights do not fit one {GPU_MEM[0]:.0f} GB GPU -> skipped"); return None
    n_rows = NTR + NTE
    est = spec["t4_min_per_1k"] * n_rows / 1000 * (1 if on_gpu else 30) + 1.5 + (4 if CFG["RUN_CV"] else 0)
    if not budget_ok(est, f"B[{key}]"):
        return None
    t = time.time()
    with gpu_section(f"ntl-llm-{key}"):
        X = extract_mean_feats(path, tr_texts + te_texts, spec["layers"], bs=spec["bs"], on_gpu=on_gpu)
    log(f"B[{key}]: features {X.shape} in {time.time() - t:.0f}s")
    Xtr, Xte = X[:NTR], X[NTR:]
    del X
    hier = key == "q4b" and CFG["Q4B_HIER"]
    nc, sh = spec["ncomp"], spec["shrink"]
    comp = dict(test=None, test_raw=None, oof=None)
    Xtr_c = group_center(Xtr, gtr)
    T = spec["T"]
    # ---- in-notebook OOF (val rows centred by their own predicted clusters)
    if CFG["RUN_CV"] and oof_groups is not None:
        Zo = np.zeros((NTR, NC)); Zso = [np.zeros((NTR, len(fi))) for fi in FAMI]
        for f in FOLD_IDS:
            tr_i, va_i = np.where(folds != f)[0], np.where(folds == f)[0]
            Xva_c = group_center(Xtr[va_i], oof_groups[va_i])
            if hier:
                zg, zs = hier_decisions(Xtr_c[tr_i], y_tr[tr_i], [Xva_c], nc, sh)
                Zo[va_i] = zg[0]
                for k in range(len(FAMI)):
                    Zso[k][va_i] = zs[k][0]
            else:
                Zo[va_i] = lda_full(Xtr_c[tr_i], y_tr[tr_i], [Xva_c], nc, sh)[0]
            log(f"  B[{key}] fold {f}: bacc {bacc(y_tr[va_i], Zo[va_i]):.4f}")
        T = fit_T_logits(Zo, y_tr, spec["T"])
        comp["oof"] = hier_proba(Zo, Zso, Tg=T) if hier else softmax(Zo / T)
        log(f"B[{key}]: OOF bacc = {bacc(y_tr, comp['oof']):.4f} (fitted T={T:.2f})")
    # ---- final fit on all train rows, test centred by predicted clusters
    Xte_c = group_center(Xte, test_groups)
    if hier:
        zg, zs = hier_decisions(Xtr_c, y_tr, [Xte_c], nc, sh)
        comp["test"] = hier_proba(zg[0], [z[0] for z in zs], Tg=T)
    else:
        comp["test"] = softmax(lda_full(Xtr_c, y_tr, [Xte_c], nc, sh)[0] / T)
    del Xtr_c, Xte_c
    # ---- uncentred twin for rows in tiny clusters
    if (ALPHA_TE < 1).any():
        comp["test_raw"] = softmax(lda_full(Xtr, y_tr, [Xte], nc, sh)[0] / spec["T_raw"])
    del Xtr, Xte
    gc.collect()
    log(f"B[{key}]: done in {time.time() - t:.0f}s")
    return comp

In [ ]:
if CFG["USE_LLM"]:
    for _key in CFG["LLM_MODELS"]:
        with section(f"B: frozen LM {_key}"):
            _c = run_llm_component(_key)
            if _c is not None and _c["test"] is not None:
                COMP[_key] = _c
        free_gpu()
log(f"components available so far: {list(COMP)}")

## 6. Component C: DeBERTa-v3-large, group-aware fine-tune

**Why:** a fine-tuned encoder is weaker on its own (honest CV 0.458) but errs differently from the frozen LMs.
It is the **most valuable add-on** in the final blend: dropping it costs 0.009 decoded balanced accuracy.

**Recipe.** A plain sequence classifier only reaches 0.33. What works is training on **whole prompt groups**:
* Each step takes 2 prompt groups and keeps a random 50-100% of each group's passages.
* The mean-pooled embedding feeds two heads. The **raw** head sees `h`. The **centred** head sees
  `h - mean_group(h)`.
* A **column loss** (weight 0.5): for each model present in a group, a softmax over that group's passages. This
  mimics the assignment step.
* Training: AdamW, backbone lr 2e-5, head lr 5e-4, 7 epochs, 10% warm-up, fp16 autocast + GradScaler
  (T4-safe), non-reentrant gradient checkpointing (the reentrant version crashes DeBERTa-v2).

At test time the centred head uses our predicted clusters, and the raw head is the tiny-cluster fallback.
In-notebook CV would cost 5 x 45 min, so this component is validated offline only. It is not part of the
in-notebook OOF blend.

In [ ]:
def run_deberta():
    import torch.nn as nn
    import torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
    path = MODEL_PATHS.get("deberta")
    if not path:
        log("C: deberta-v3-large not found -> skipped"); return None
    if not HAS_CUDA:
        log("C: no GPU -> skipped"); return None
    n_ep = int(CFG["DEB_EPOCHS"])
    steps_per_ep = math.ceil(len(np.unique(gtr)) / CFG["DEB_GROUPS_PER_STEP"])
    total = steps_per_ep * n_ep
    if CFG["DEB_MAX_STEPS"]:
        total = min(total, int(CFG["DEB_MAX_STEPS"]))
    est = 6.5 * total / 75 + 0.8 * NTE / 1000 + 1      # T4: ~6.5 min per 75-step epoch, ~0.8 min / 1k test rows
    if not budget_ok(est, "C: DeBERTa"):
        return None
    seed_everything(CFG["SEED"])
    tok = AutoTokenizer.from_pretrained(path)
    enc_tr = tok(tr_texts, truncation=True, max_length=CFG["DEB_MAX_LEN"])["input_ids"]
    enc_te = tok(te_texts, truncation=True, max_length=CFG["DEB_MAX_LEN"])["input_ids"]
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0

    def collate(seqs):
        L = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), L), pad, dtype=torch.long); mask = torch.zeros((len(seqs), L), dtype=torch.long)
        for j, s in enumerate(seqs):
            ids[j, :len(s)] = torch.tensor(s); mask[j, :len(s)] = 1
        return ids.cuda(), mask.cuda()

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.bb = from_pretrained_dtype(AutoModel, path, torch.float32)
            self.bb.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            H = self.bb.config.hidden_size
            self.drop = nn.Dropout(0.1)
            self.head_raw = nn.Linear(H, NC); self.head_cen = nn.Linear(H, NC)

        def embed(self, ids, mask):
            h = self.bb(input_ids=ids, attention_mask=mask).last_hidden_state
            m = mask.unsqueeze(-1).to(h.dtype)
            return ((h * m).sum(1) / m.sum(1).clamp(min=1)).float()

        def heads(self, h, gid):
            G = int(gid.max()) + 1
            mu = torch.zeros(G, h.shape[1], device=h.device).index_add_(0, gid, h) / \
                torch.zeros(G, device=h.device).index_add_(0, gid, torch.ones_like(gid, dtype=torch.float))[:, None]
            return self.head_raw(self.drop(h)), self.head_cen(self.drop(h - mu[gid]))

    t0 = time.time()
    with gpu_section("ntl-deberta"):
        model = Net().cuda()
        nd = lambda n, p_: p_.ndim < 2 or "norm" in n.lower() or n.endswith(".bias")  # noqa
        pg = [{"params": [p_ for n, p_ in model.named_parameters() if n.startswith("head")], "lr": CFG["DEB_HEAD_LR"], "weight_decay": 0.0},
              {"params": [p_ for n, p_ in model.named_parameters() if not n.startswith("head") and not nd(n, p_)], "lr": CFG["DEB_LR"], "weight_decay": 0.01},
              {"params": [p_ for n, p_ in model.named_parameters() if not n.startswith("head") and nd(n, p_)], "lr": CFG["DEB_LR"], "weight_decay": 0.0}]
        opt = torch.optim.AdamW(pg, betas=(0.9, 0.98), eps=1e-6)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
        try:
            scaler = torch.amp.GradScaler("cuda")
        except Exception:
            scaler = torch.cuda.amp.GradScaler()
        groups = [np.where(gtr == g)[0] for g in np.unique(gtr)]
        step = 0
        for ep in range(n_ep):
            model.train()
            gp = np.random.permutation(len(groups))
            for b in range(0, len(gp), CFG["DEB_GROUPS_PER_STEP"]):
                if step >= total:
                    break
                idx, gid = [], []
                for gi, g in enumerate(gp[b:b + CFG["DEB_GROUPS_PER_STEP"]]):
                    mem = groups[g]
                    k = max(4, int(round(len(mem) * random.uniform(CFG["DEB_KEEP_MIN"], 1.0))))
                    sel = np.random.choice(mem, size=min(k, len(mem)), replace=False)
                    idx += sel.tolist(); gid += [gi] * len(sel)
                gid = torch.tensor(gid).cuda(); yy = torch.tensor(y_tr[idx]).cuda()
                with torch.autocast("cuda", dtype=torch.float16):
                    h = model.embed(*collate([enc_tr[i] for i in idx]))
                lr_, lc_ = model.heads(h.float(), gid)
                loss = F.cross_entropy(lr_, yy) + F.cross_entropy(lc_, yy)
                if CFG["DEB_COL_W"] > 0:
                    cl = 0
                    for gi in range(int(gid.max()) + 1):
                        rows = (gid == gi).nonzero().squeeze(1)
                        cl = cl + F.cross_entropy(lc_[rows][:, yy[rows]].T, torch.arange(len(rows), device=yy.device), reduction="sum")
                    loss = loss + CFG["DEB_COL_W"] * cl / len(idx)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()
                step += 1
            log(f"  C: epoch {ep + 1}/{n_ep} done ({step} steps, loss {loss.item():.3f}, {time.time() - t0:.0f}s)")
            if step >= total:
                break
        # ---- test inference
        model.eval()
        order = np.argsort([len(e) for e in enc_te])
        Hs = []
        with torch.no_grad():
            for b in range(0, len(order), CFG["DEB_EVAL_BS"]):
                with torch.autocast("cuda", dtype=torch.float16):
                    Hs.append(model.embed(*collate([enc_te[i] for i in order[b:b + CFG["DEB_EVAL_BS"]]])).float())
            Hc = torch.cat(Hs)
            H = torch.empty_like(Hc); H[torch.as_tensor(order, device=Hc.device)] = Hc
            lr_, lc_ = model.heads(H, torch.as_tensor(test_groups, device=H.device))
            comp = dict(test=torch.softmax(lc_, 1).cpu().numpy().astype(np.float64),
                        test_raw=torch.softmax(lr_, 1).cpu().numpy().astype(np.float64), oof=None)
        del model, opt, sch, scaler, H, Hc, Hs, lr_, lc_
    free_gpu()
    log(f"C: done in {time.time() - t0:.0f}s")
    return comp

In [ ]:
if CFG["USE_DEBERTA"]:
    with section("C: DeBERTa-v3-large"):
        _c = run_deberta()
        if _c is not None:
            COMP["deberta"] = _c
    free_gpu()
log(f"components available: {list(COMP)}")

## 7. Ensemble and group decoding

The settings below come from an honest nested CV: predicted clusters, out-of-fold pair models, and weights and
decoder chosen on 4 folds and scored on the 5th.
1. **Blend: an equal-weight mean of the centred log-probs** of the 5 default components (q4b-hier, q17, q06,
   TF, DeBERTa), plus any optional LM that is enabled. Fitted weights did not generalise: nested selection gave 0.716 +- 0.014, and every
   candidate landed in 0.715-0.721. Equal weights have zero free parameters and were picked in 3 of 5 outer
   folds. A missing component simply drops out of the mean.
2. **Tiny-cluster fallback ("mix9").** For a passage in a predicted cluster of size `m`:
   `Z = alpha Z_centred + (1 - alpha) Z_raw`, with `alpha = clip((m - 1) / 8, 0, 1)`.
   `Z_raw` is a weighted mean of the uncentred twins (4B 1, 8B/1.7B/SmolLM2/OLMo 0.5, 0.6B/TF/DeBERTa 0.3).
   Accuracy by cluster size goes from 0.015 to 0.50 for singletons, 0.39 to 0.50 for size 2-3, and 0.50 to 0.54
   for size 4-8. Full groups are unchanged.
3. **Decoding inside each predicted cluster:**
   * Sinkhorn marginals with each model at most once per cluster (`col='le'`), at `power = 2.05`
     (1.5 x the calibrated 1/T of the blend).
   * A **pairwise sibling term** (`lam = 0.25`, mean-field). Inside a prompt group, the within-group z-scored
     similarity of two passages depends strongly on *which two models* wrote them (fold-to-fold correlation 0.92).
     Examples: gemini-3.7 ~ 3.8-flash, claude-fable-5 ~ opus-4-8 ~ glm, gpt-5.6-sol ~ terra. We model it as a
     Gaussian over 4 channels (word 1-3, char 3-5, typography vector, passage opening), fitted on the train
     groups.
   * Nested CV picked x1.5 / lam 0.25 in 5 of 5 outer folds. The comparison is in the table below.

| decoder (honest CV, md0 = full groups) | full | 20% per-model dropout | 40% dropout |
|---|---|---|---|
| argmax | 0.617 | 0.619 | 0.616 |
| Hungarian | 0.721 | 0.702 | 0.680 |
| Sinkhorn, no pair term | 0.727 | 0.705 | 0.681 |
| **Sinkhorn + pair (x1.5, lam 0.25), final** | **0.737-0.740** | **0.714** | **0.692** |

The first three rows come from the decoder comparison, which used provisional weights. The final row gives the
range of the final blend and its nested-CV mean.

4. With `RUN_CV`, the same decoding runs on the in-notebook OOF. Each validation fold is clustered on its own
   and the pair models are fitted out-of-fold. DeBERTa has no in-notebook OOF, so the OOF blend is the mean
   without it.

In [ ]:
def logp(P):
    return np.log(np.clip(np.asarray(P, float), 1e-12, None))


def blend_logits(comps, weights, key):
    """Normalised weighted mean of log-probs over the components that have `key` ('test', 'test_raw' or 'oof')."""
    ws = {k: float(w) for k, w in weights.items() if float(w) > 0 and k in comps and comps[k].get(key) is not None}
    if not ws:
        return None, {}
    tot = sum(ws.values())
    Z = sum(w * logp(comps[k][key]) for k, w in ws.items()) / tot
    return Z, {k: round(w / tot, 3) for k, w in ws.items()}


def mix_small(Z_pc, Z_raw, alpha):
    if Z_raw is None or not (alpha < 1).any():
        return Z_pc
    a = np.asarray(alpha, float)[:, None]
    return a * Z_pc + (1 - a) * Z_raw


def merged_test(c, alpha=None):
    """Single component with the same small-cluster rule (for per-component reporting)."""
    alpha = ALPHA_TE if alpha is None else alpha
    Z = logp(c["test"])
    if c.get("test_raw") is not None:
        Z = mix_small(Z, logp(c["test_raw"]), alpha)
    return softmax(Z)


def sanitize_components(comps):
    """A NaN/inf row (e.g. an fp16 overflow) would make every passage of its cluster decode to class 0.
    Repair isolated bad rows with a uniform distribution; drop a component whose matrix is mostly broken."""
    for k in list(comps):
        for key in ("test", "test_raw", "oof"):
            P = comps[k].get(key)
            if P is None:
                continue
            P = np.asarray(P, float)
            bad = ~np.isfinite(P).all(1) | (P.sum(1) <= 0)
            if not bad.any():
                continue
            if bad.mean() > 0.05:
                log(f"!!! component {k}/{key}: {int(bad.sum())} non-finite rows -> dropped")
                comps[k][key] = None
            else:
                P = P.copy()
                P[bad] = 1.0 / NC
                comps[k][key] = P / P.sum(1, keepdims=True)
                log(f"component {k}/{key}: {int(bad.sum())} non-finite rows replaced by uniform")
        if comps[k].get("test") is None and comps[k].get("test_raw") is None:
            comps.pop(k)


sanitize_components(COMP)
W_PC = {k: float(v) for k, v in CFG["ENS_WEIGHTS"].items() if float(v) > 0}
if not any(k in COMP for k in W_PC) and COMP:          # nothing weighted survived -> equal weights over what exists
    log("no weighted component available -> equal weights over available components")
    W_PC = {k: 1.0 for k in COMP}
W_RAW = {k: float(v) for k, v in CFG["RAW_WEIGHTS"].items() if float(v) > 0}

P_TEST, P_OOF, USED_W, USED_W_RAW, USED_W_OOF = None, None, {}, {}, {}
with section("blend"):
    Z_pc, USED_W = blend_logits(COMP, W_PC, "test")
    Z_raw, USED_W_RAW = blend_logits(COMP, W_RAW, "test_raw")
    if Z_pc is not None:
        P_TEST = softmax(mix_small(Z_pc, Z_raw, ALPHA_TE))
    elif Z_raw is not None:
        P_TEST = softmax(Z_raw)
    log(f"test blend (centred) weights: {USED_W}")
    log(f"raw-twin weights (used for {int((ALPHA_TE < 1).sum())} rows in clusters < {CFG['SMALL_DENOM'] + 1}): {USED_W_RAW}")
    if CFG["RUN_CV"]:
        Z_oof, USED_W_OOF = blend_logits(COMP, W_PC, "oof")
        if Z_oof is not None:
            P_OOF = softmax(Z_oof)
            log(f"OOF blend weights: {USED_W_OOF} -> OOF argmax bacc {bacc(y_tr, P_OOF):.4f}")
            for k, c in COMP.items():
                if c.get("oof") is not None:
                    log(f"   OOF {k:8s} argmax bacc {bacc(y_tr, c['oof']):.4f}")

# ---- decoding power
POWER = CFG["DECODE"].get("power")
# 2.05 was calibrated for the default blend. If any of its components is missing, the blend's sharpness changes:
# refit power_mult x 1/T on the in-notebook OOF blend instead (its component set then matches the test blend
# whenever DeBERTa, the only component without an in-notebook OOF, is the one missing).
RECIPE_SET = {"q4b", "q17", "q06", "tf", "deberta"}
if POWER is not None and not RECIPE_SET.issubset(USED_W) and P_OOF is not None:
    log(f"components {sorted(RECIPE_SET - set(USED_W))} missing -> decoding power refitted on the OOF blend")
    POWER = None
try:
    if POWER is None and P_OOF is not None:
        POWER = CFG["DECODE"]["power_mult"] * fit_temperature(P_OOF, y_tr)
        log(f"decoding power = {CFG['DECODE']['power_mult']} x fitted 1/T = {POWER:.3f}")
except Exception as e:  # noqa
    log(f"power fit failed: {e}")
    POWER = None
if POWER is None:   # calibrated 1/T of the honest offline blend ~ 1.32
    POWER = CFG["DECODE"]["power_mult"] * 1.32
    log(f"decoding power (default) = {POWER:.3f}")
else:
    log(f"decoding power = {POWER:.3f}")
if P_OOF is not None:
    log(f"(info) in-notebook OOF blend calibrated 1/T = {fit_temperature(P_OOF, y_tr):.3f}; offline recipe: 1.32")

PAIR_MODEL, S_TR = None, None
with section("fit pair model on train groups"):
    PAIR_MODEL, S_TR = fit_pair_model_from_texts(tr_texts, y_tr, gtr)

In [ ]:
PRED_TEST, HOW = None, "none"
with section("decode test"):
    if P_TEST is not None:
        if not np.isfinite(P_TEST).all():
            log("!!! non-finite blend probabilities -> repaired")
            P_TEST = np.nan_to_num(P_TEST, nan=1.0 / NC, posinf=1.0, neginf=0.0) + 1e-12
            P_TEST = P_TEST / P_TEST.sum(1, keepdims=True)
        (PRED_TEST, Q_TEST), HOW = decode_groups(P_TEST, te_texts, test_groups, PAIR_MODEL, CFG["DECODE"], POWER)
        log(f"test decoded with {HOW}; changed {int((PRED_TEST != P_TEST.argmax(1)).sum())} / {NTE} argmax labels")

In [ ]:
PRED_OOF = None
with section("decoded OOF evaluation"):
    if P_OOF is not None and oof_groups is not None:
        blocks_true = pair_blocks(S_TR, gtr)
        res = {"argmax": bacc(y_tr, P_OOF)}
        pred_h = assign(P_OOF, oof_groups, method="hungarian")
        res["hungarian (pred clusters)"] = bacc(y_tr, pred_h)
        pred_s = assign(P_OOF, oof_groups, method="sinkhorn", power=POWER, col=CFG["DECODE"]["col"])
        res["sinkhorn"] = bacc(y_tr, pred_s)
        PRED_OOF = np.zeros(NTR, int)
        for f in FOLD_IDS:
            va = np.where(folds == f)[0]
            pm_f = fit_pair_model({g: b for g, b in blocks_true.items() if folds[b[0][0]] != f}, y_tr)
            S_va = pair_similarities([tr_texts[i] for i in va])
            PRED_OOF[va] = assign(P_OOF[va], oof_groups[va], pair=(pair_blocks(S_va, oof_groups[va]), pm_f),
                                  method="sinkhorn", power=POWER, col=CFG["DECODE"]["col"],
                                  pair_lam=CFG["DECODE"]["pair_lam"], mf_iter=CFG["DECODE"]["mf_iter"],
                                  mf_damp=CFG["DECODE"]["mf_damp"], final=CFG["DECODE"]["final"])
        res["sinkhorn + pair (final)"] = bacc(y_tr, PRED_OOF)
        print("In-notebook OOF (val folds clustered on their own, pair models out-of-fold; "
              f"components {list(USED_W_OOF)}):")
        display(pd.DataFrame({"balanced accuracy": res}).round(4))

In [ ]:
with section("per-class recall + confusion"):
    _y, _p, _what = None, None, ""
    if PRED_OOF is not None:
        _y, _p, _what = y_tr, PRED_OOF, "decoded OOF"
    elif Y_HOLDOUT is not None and PRED_TEST is not None:
        _y, _p, _what = Y_HOLDOUT, PRED_TEST, "decoded holdout"
    if _y is not None:
        rec = pd.Series(confusion_matrix(_y, _p, labels=range(NC)).diagonal() / np.bincount(_y, minlength=NC).clip(1),
                        index=LABELS).sort_values()
        print(f"Per-class recall ({_what}):")
        display(rec.round(3).to_frame("recall").T)
        if HAS_PLOT:
            cm = confusion_matrix(_y, _p, labels=range(NC), normalize="true")
            plt.figure(figsize=(13, 11))
            sns.heatmap(pd.DataFrame(cm, index=LABELS, columns=LABELS), cmap="Blues", annot=True, fmt=".2f",
                        annot_kws={"size": 7}, cbar=False)
            plt.title(f"Confusion matrix, row-normalised ({_what})"); plt.ylabel("true"); plt.xlabel("predicted")
            plt.tight_layout(); plt.show()

The remaining errors are almost all **sibling** confusions: glm-5.3 vs glm-5.3-flash (near chance), claude-fable-5
vs glm, gemini-3.7 vs 3.8-flash, gpt-5.6-sol vs terra. Vendor families are close to solved.

## 8. Submission

`submission.parquet` (columns `id`, `model`) goes to the working directory. The fallback chain is: decoded blend,
then blend argmax, then TF-IDF alone, then any component, then a constant. The file is validated before it is
written: every test id appears exactly once, there are no nulls, every label is one of the 18 models, and the row
count equals the test row count.

In [ ]:
def make_labels():
    if PRED_TEST is not None and len(PRED_TEST) == NTE:
        return np.asarray(PRED_TEST), f"blend + {HOW}"
    if P_TEST is not None:
        return P_TEST.argmax(1), "blend argmax"
    for k in ["tf"] + list(COMP):
        if k in COMP:
            try:
                return merged_test(COMP[k]).argmax(1), f"{k} argmax"
            except Exception:
                pass
    return np.full(NTE, L2I["claude-opus-4-6"]), "constant"


try:
    pred_idx, SOURCE = make_labels()
except Exception as e:  # noqa
    log(f"label fallback: {e}")
    pred_idx, SOURCE = np.full(NTE, 0), "constant"
sub = pd.DataFrame({"id": test["id"].astype(str).values, "model": [LABELS[int(i)] for i in pred_idx]})

ok = (len(sub) == NTE and sub["id"].is_unique and set(sub["id"]) == set(test["id"].astype(str))
      and sub.notna().all().all() and sub["model"].isin(LABELS).all())
if not ok:
    log("!!! submission failed validation -> constant fallback")
    sub = pd.DataFrame({"id": test["id"].astype(str).values, "model": LABELS[0]})
    SOURCE = "constant (validation fallback)"
try:
    sub.to_parquet("submission.parquet", index=False)
except Exception as e:  # noqa  (pyarrow missing?) -> try the other engine
    log(f"to_parquet failed ({e}); retrying with fastparquet")
    sub.to_parquet("submission.parquet", index=False, engine="fastparquet")
chk = pd.read_parquet("submission.parquet")
_valid = (len(chk) == NTE and chk["id"].is_unique and chk["model"].isin(LABELS).all() and chk.notna().all().all())
log(f"wrote submission.parquet: {chk.shape} | valid={_valid} | source: {SOURCE} | cwd: {os.getcwd()}")
display(chk.head())
print(chk["model"].value_counts().to_string())

In [ ]:
with section("holdout report"):
    if Y_HOLDOUT is not None:
        rows = {}
        for k, c in COMP.items():
            rows[f"{k} (argmax)"] = bacc(Y_HOLDOUT, merged_test(c))
        if P_TEST is not None:
            rows["blend (argmax)"] = bacc(Y_HOLDOUT, P_TEST)
            rows["blend + Hungarian (pred clusters)"] = bacc(Y_HOLDOUT, assign(P_TEST, test_groups, method="hungarian"))
        rows[f"FINAL submission ({SOURCE})"] = balanced_accuracy_score(Y_HOLDOUT, chk["model"].map(L2I).values)
        rows["test clustering ARI"] = cluster_report(test_groups, PID_HOLDOUT)["ari"]
        print(f"LOCAL HOLDOUT (train on folds != {CFG['HOLDOUT_FOLD']}, predict fold {CFG['HOLDOUT_FOLD']} "
              f"blind, {NTE} passages):")
        display(pd.DataFrame({"balanced accuracy": rows}).round(4))
        HOLDOUT_FINAL = rows[f"FINAL submission ({SOURCE})"]

print("Section timings (s):", {k: round(v, 1) for k, v in SECTION_TIMES.items()})
log(f"TOTAL elapsed: {(time.time() - T0) / 60:.1f} min")